# CReST: Final Cross-System Training

Trains the locked architecture across seven outer folds with whole-system train, validation, and test separation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Final CReST cross-system training environment

from pathlib import Path
import json
import random
import gc
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
    Subset
)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

import matplotlib.pyplot as plt


# Paths

BASE_DIR = Path.cwd()

HANDOFF_DIR = (
    BASE_DIR
    / "processed"
    / "crest_v5_handoff"
)

CALIBRATION_DIR = (
    BASE_DIR
    / "processed"
    / "crest_design_calibration"
)

CHECKPOINT_ROOT = (
    BASE_DIR
    / "checkpoints"
    / "final_7fold"
)

RESULTS_DIR = (
    BASE_DIR
    / "results"
)

FIG_DIR = (
    BASE_DIR
    / "figures"
)

ARCHITECTURE_LOCK_PATH = (
    RESULTS_DIR
    / "crest_final_architecture_lock.json"
)


CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    exist_ok=True
)

FIG_DIR.mkdir(
    exist_ok=True
)


# Configuration

with open(
    HANDOFF_DIR / "handoff_config.json",
    "r",
    encoding="utf-8"
) as f:

    CONFIG = json.load(f)


with open(
    ARCHITECTURE_LOCK_PATH,
    "r",
    encoding="utf-8"
) as f:

    ARCHITECTURE_LOCK = json.load(f)


assert (
    ARCHITECTURE_LOCK[
        "architecture_status"
    ]
    == "LOCKED"
)

assert (
    ARCHITECTURE_LOCK[
        "architecture_search"
    ]
    == "CLOSED"
)


GLOBAL_SEED = int(
    CONFIG["global_seed"]
)


def set_global_seed(
    seed=GLOBAL_SEED
):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


set_global_seed()


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Working directory:",
    BASE_DIR
)

print(
    "Architecture:",
    ARCHITECTURE_LOCK[
        "architecture_name"
    ]
)

print(
    "Architecture status:",
    ARCHITECTURE_LOCK[
        "architecture_status"
    ]
)

print(
    "Architecture search:",
    ARCHITECTURE_LOCK[
        "architecture_search"
    ]
)

print(
    "Device:",
    device
)

print(
    "\nEnvironment initialised."
)

Working directory: PROJECT_ROOT
Architecture: CReST-Final
Architecture status: LOCKED
Architecture search: CLOSED
Device: cpu

Final 7-fold confirmatory training notebook ready.


In [ ]:
# Load core data for final 7-fold training

# Core arrays

credit_tokens = np.load(
    HANDOFF_DIR / "credit_tokens.npy",
    mmap_mode="r"
)

behaviour_tokens = np.load(
    HANDOFF_DIR / "behaviour_tokens.npy",
    mmap_mode="r"
)

performance_mask = np.load(
    HANDOFF_DIR / "performance_mask.npy",
    mmap_mode="r"
)

interaction_mask = np.load(
    HANDOFF_DIR / "interaction_mask.npy",
    mmap_mode="r"
)

student_design_mask = np.load(
    HANDOFF_DIR / "student_design_mask.npy",
    mmap_mode="r"
)

process_tensor = np.load(
    HANDOFF_DIR / "process_tensor.npy",
    mmap_mode="r"
)

booklet_nuisance_ids = np.load(
    HANDOFF_DIR / "booklet_nuisance_ids.npy",
    mmap_mode="r"
)

item_domain_ids = np.load(
    HANDOFF_DIR / "item_domain_ids.npy",
    mmap_mode="r"
)

item_ideation_ids = np.load(
    HANDOFF_DIR / "item_ideation_ids.npy",
    mmap_mode="r"
)


# Metadata / context

primary_meta = pd.read_pickle(
    HANDOFF_DIR / "primary_meta.pkl"
)

model_context = pd.read_pickle(
    HANDOFF_DIR / "model_context.pkl"
)

context_registry_locked = pd.read_csv(
    HANDOFF_DIR / "context_registry_locked.csv"
)


# Structural assessment-design mapping
#
# This is NOT a Fold-0 response-derived calibration.
# It simply maps the 30 known booklet designs to the
# 10 recurring assessment-design signatures.

BOOKLET_TO_SIGNATURE = np.load(
    CALIBRATION_DIR
    / "booklet_to_signature.npy"
)


design_signature_ids = (
    BOOKLET_TO_SIGNATURE[
        booklet_nuisance_ids
    ]
)


# Basic audit

N_STUDENTS = len(
    primary_meta
)

N_ITEMS = (
    credit_tokens.shape[1]
)


assert (
    credit_tokens.shape
    == (142564, 32)
)

assert (
    process_tensor.shape
    == (142564, 32, 3)
)

assert (
    BOOKLET_TO_SIGNATURE.shape
    == (30,)
)

assert (
    len(
        np.unique(
            design_signature_ids
        )
    )
    == 10
)

assert (
    primary_meta[
        "CNT"
    ].nunique()
    == 63
)


print(
    "Students:",
    N_STUDENTS
)

print(
    "Items:",
    N_ITEMS
)

print(
    "Education systems:",
    primary_meta[
        "CNT"
    ].nunique()
)

print(
    "Booklets:",
    len(
        np.unique(
            booklet_nuisance_ids
        )
    )
)

print(
    "Assessment-design signatures:",
    len(
        np.unique(
            design_signature_ids
        )
    )
)

print(
    "\nCore final-training data loaded."
)

Students: 142564
Items: 32
Education systems: 63
Booklets: 30
Assessment-design signatures: 10

Core final-training data loaded.


In [ ]:
# Locked 7-fold cross-system protocol

N_COUNTRY_FOLDS = 7


def get_outer_split(
    outer_fold
):

    test_fold = int(
        outer_fold
    )

    validation_fold = (
        test_fold + 1
    ) % N_COUNTRY_FOLDS


    fold_values = (
        primary_meta[
            "country_fold"
        ]
        .to_numpy()
    )


    train_idx = np.where(

        ~np.isin(
            fold_values,
            [
                test_fold,
                validation_fold
            ]
        )

    )[0]


    validation_idx = np.where(
        fold_values
        == validation_fold
    )[0]


    test_idx = np.where(
        fold_values
        == test_fold
    )[0]


    return {

        "outer_fold":
            test_fold,

        "test_fold":
            test_fold,

        "validation_fold":
            validation_fold,

        "train_idx":
            train_idx,

        "validation_idx":
            validation_idx,

        "test_idx":
            test_idx
    }


# Audit all 7 folds

protocol_rows = []


for outer_fold in range(
    N_COUNTRY_FOLDS
):

    split = get_outer_split(
        outer_fold
    )


    train_countries = sorted(
        primary_meta.iloc[
            split[
                "train_idx"
            ]
        ]["CNT"]
        .unique()
        .tolist()
    )


    validation_countries = sorted(
        primary_meta.iloc[
            split[
                "validation_idx"
            ]
        ]["CNT"]
        .unique()
        .tolist()
    )


    test_countries = sorted(
        primary_meta.iloc[
            split[
                "test_idx"
            ]
        ]["CNT"]
        .unique()
        .tolist()
    )


    # No system overlap allowed
    assert not (
        set(train_countries)
        &
        set(validation_countries)
    )

    assert not (
        set(train_countries)
        &
        set(test_countries)
    )

    assert not (
        set(validation_countries)
        &
        set(test_countries)
    )


    assert (
        len(train_countries)
        == 45
    )

    assert (
        len(validation_countries)
        == 9
    )

    assert (
        len(test_countries)
        == 9
    )


    protocol_rows.append(
        {
            "outer_fold":
                outer_fold,

            "train_n":
                len(
                    split[
                        "train_idx"
                    ]
                ),

            "validation_n":
                len(
                    split[
                        "validation_idx"
                    ]
                ),

            "test_n":
                len(
                    split[
                        "test_idx"
                    ]
                ),

            "train_systems":
                len(
                    train_countries
                ),

            "validation_systems":
                len(
                    validation_countries
                ),

            "test_systems":
                len(
                    test_countries
                ),

            "validation_fold":
                split[
                    "validation_fold"
                ],

            "test_fold":
                split[
                    "test_fold"
                ]
        }
    )


final_protocol_df = pd.DataFrame(
    protocol_rows
)


print(
    final_protocol_df.to_string(
        index=False
    )
)


# Every system must appear exactly once as TEST

test_country_occurrences = {}


for outer_fold in range(
    N_COUNTRY_FOLDS
):

    split = get_outer_split(
        outer_fold
    )


    countries = (
        primary_meta.iloc[
            split[
                "test_idx"
            ]
        ]["CNT"]
        .unique()
    )


    for country in countries:

        test_country_occurrences[
            country
        ] = (
            test_country_occurrences.get(
                country,
                0
            )
            + 1
        )


assert (
    len(
        test_country_occurrences
    )
    == 63
)

assert all(
    count == 1
    for count
    in test_country_occurrences.values()
)


print(
    "\nEvery education system appears "
    "exactly once as held-out TEST: PASS"
)

print(
    "7-fold confirmatory protocol: LOCKED"
)

 outer_fold  train_n  validation_n  test_n  train_systems  validation_systems  test_systems  validation_fold  test_fold
          0    99966         20949   21649             45                   9             9                1          0
          1   100846         20769   20949             45                   9             9                2          1
          2   101471         20324   20769             45                   9             9                3          2
          3   102780         19460   20324             45                   9             9                4          3
          4   103410         19694   19460             45                   9             9                5          4
          5   103151         19719   19694             45                   9             9                6          5
          6   101196         21649   19719             45                   9             9                0          6

Every education system appears exactly 

In [ ]:
# Leakage-safe fold-specific preprocessors

common_student_vars = list(
    CONFIG["common_student_vars"]
)

common_school_vars = list(
    CONFIG["common_school_vars"]
)


class ContextPreprocessor:

    def __init__(
        self,
        registry,
        variables
    ):

        reg = registry.loc[
            registry["variable"].isin(
                variables
            )
        ].copy()


        self.continuous_vars = (
            reg.loc[
                reg["feature_type"] == "continuous",
                "variable"
            ]
            .tolist()
        )


        self.categorical_vars = (
            reg.loc[
                reg["feature_type"].isin(
                    [
                        "categorical",
                        "binary",
                        "ordinal"
                    ]
                ),
                "variable"
            ]
            .tolist()
        )


        self.medians = {}
        self.iqrs = {}
        self.category_maps = {}


    def fit(
        self,
        df,
        indices
    ):

        train_df = df.iloc[
            indices
        ]


        # Continuous
        for var in self.continuous_vars:

            x = pd.to_numeric(
                train_df[var],
                errors="coerce"
            )


            median = x.median()

            iqr = (
                x.quantile(0.75)
                -
                x.quantile(0.25)
            )


            if (
                not np.isfinite(iqr)
                or
                iqr < 1e-8
            ):
                iqr = 1.0


            self.medians[var] = float(
                median
            )

            self.iqrs[var] = float(
                iqr
            )


        # Categorical
        for var in self.categorical_vars:

            values = sorted(
                train_df[var]
                .dropna()
                .unique()
                .tolist(),
                key=str
            )


            # 0 = missing
            # 1 = unseen
            # 2+ = categories seen in training systems
            self.category_maps[var] = {
                value: i + 2
                for i, value
                in enumerate(values)
            }


        return self


    def transform(
        self,
        df
    ):

        n = len(df)


        # Continuous

        cont_values = []
        cont_masks = []


        for var in self.continuous_vars:

            x = pd.to_numeric(
                df[var],
                errors="coerce"
            ).to_numpy(
                dtype=np.float32
            )


            observed = np.isfinite(
                x
            )


            z = (
                x
                -
                self.medians[var]
            ) / self.iqrs[var]


            z[
                ~observed
            ] = 0.0


            cont_values.append(
                z
            )

            cont_masks.append(
                observed
            )


        continuous = (
            np.column_stack(
                cont_values
            ).astype(
                np.float32
            )
            if cont_values
            else
            np.empty(
                (n, 0),
                dtype=np.float32
            )
        )


        continuous_mask = (
            np.column_stack(
                cont_masks
            )
            if cont_masks
            else
            np.empty(
                (n, 0),
                dtype=bool
            )
        )


        # Categorical

        cat_values = []


        for var in self.categorical_vars:

            mapping = (
                self.category_maps[var]
            )


            raw = df[var].to_numpy()


            encoded = np.zeros(
                n,
                dtype=np.int64
            )


            for i, value in enumerate(
                raw
            ):

                if pd.isna(value):

                    encoded[i] = 0

                else:

                    encoded[i] = (
                        mapping.get(
                            value,
                            1
                        )
                    )


            cat_values.append(
                encoded
            )


        categorical = (
            np.column_stack(
                cat_values
            )
            if cat_values
            else
            np.empty(
                (n, 0),
                dtype=np.int64
            )
        )


        return {

            "continuous":
                continuous,

            "continuous_mask":
                continuous_mask,

            "categorical":
                categorical
        }


    def cardinalities(
        self
    ):

        return [
            len(
                self.category_maps[var]
            )
            + 2

            for var
            in self.categorical_vars
        ]


class ProcessPreprocessor:

    def fit(
        self,
        x_train
    ):

        self.median = np.nanmedian(
            x_train,
            axis=0
        )


        q25 = np.nanpercentile(
            x_train,
            25,
            axis=0
        )


        q75 = np.nanpercentile(
            x_train,
            75,
            axis=0
        )


        self.iqr = (
            q75 - q25
        )


        self.median[
            ~np.isfinite(
                self.median
            )
        ] = 0.0


        self.iqr[
            ~np.isfinite(
                self.iqr
            )
        ] = 1.0


        self.iqr[
            self.iqr < 1e-8
        ] = 1.0


        return self


    def transform(
        self,
        x
    ):

        observed = np.isfinite(
            x
        )


        scaled = (
            x
            -
            self.median[
                None, :, :
            ]
        ) / self.iqr[
            None, :, :
        ]


        scaled = np.asarray(
            scaled,
            dtype=np.float32
        )


        scaled[
            ~observed
        ] = 0.0


        return (
            scaled,
            observed
        )


print(
    "Fold-specific preprocessing classes ready."
)

Fold-specific preprocessing classes ready.


In [ ]:
# Fold-specific item residualization utilities

N_ITEMS = 32
N_PROCESSES = 3
N_PROCESS_CHANNELS = 3

EPS = 1e-6


process_ids_np = np.asarray(
    item_ideation_ids,
    dtype=np.int64
)


def learn_item_baselines(
    train_idx
):

    credit_item_mean = np.zeros(
        N_ITEMS,
        dtype=np.float32
    )


    behaviour_item_rates = np.zeros(
        (
            N_ITEMS,
            3
        ),
        dtype=np.float32
    )


    process_item_mean = np.zeros(
        (
            N_ITEMS,
            N_PROCESS_CHANNELS
        ),
        dtype=np.float32
    )


    process_item_std = np.ones(
        (
            N_ITEMS,
            N_PROCESS_CHANNELS
        ),
        dtype=np.float32
    )


    # Item-by-item training-only estimates

    for item in range(
        N_ITEMS
    ):

        # Credit

        credit_mask = np.asarray(
            performance_mask[
                train_idx,
                item
            ],
            dtype=bool
        )


        tokens = np.asarray(
            credit_tokens[
                train_idx,
                item
            ]
        )


        score = (
            tokens.astype(
                np.float32
            )
            - 1.0
        ) / 2.0


        if credit_mask.any():

            credit_item_mean[item] = (
                score[
                    credit_mask
                ].mean()
            )


        # Behaviour

        behaviour = np.asarray(
            behaviour_tokens[
                train_idx,
                item
            ]
        )


        behaviour_mask = np.asarray(
            interaction_mask[
                train_idx,
                item
            ],
            dtype=bool
        )


        if behaviour_mask.any():

            for k, token in enumerate(
                [1, 2, 3]
            ):

                behaviour_item_rates[
                    item,
                    k
                ] = (

                    behaviour[
                        behaviour_mask
                    ]

                    == token

                ).mean()


        # Process

        for channel in range(
            N_PROCESS_CHANNELS
        ):

            x = np.asarray(
                process_tensor[
                    train_idx,
                    item,
                    channel
                ],
                dtype=np.float32
            )


            observed = np.isfinite(
                x
            )


            if observed.any():

                values = x[
                    observed
                ]


                mean = float(
                    values.mean()
                )


                std = float(
                    values.std(
                        ddof=0
                    )
                )


                if (
                    not np.isfinite(std)
                    or
                    std < EPS
                ):

                    std = 1.0


                process_item_mean[
                    item,
                    channel
                ] = mean


                process_item_std[
                    item,
                    channel
                ] = std


    return {

        "credit_item_mean":
            credit_item_mean,

        "behaviour_item_rates":
            behaviour_item_rates,

        "process_item_mean":
            process_item_mean,

        "process_item_std":
            process_item_std
    }


# Build the exact 21-D residual representation used in the
# locked final architecture:
#
# 3 credit residual summaries
# 9 behaviour residual summaries
# 9 process residual summaries

def build_residual_summary_features(
    indices,
    baselines
):

    n = len(
        indices
    )


    credit_features = []
    behaviour_features = []
    process_features = []


    for process_id in range(
        N_PROCESSES
    ):

        selected_items = np.where(
            process_ids_np
            == process_id
        )[0]


        # Credit residual

        value_list = []
        mask_list = []


        for item in selected_items:

            tokens = np.asarray(
                credit_tokens[
                    indices,
                    item
                ]
            )


            observed = np.asarray(
                performance_mask[
                    indices,
                    item
                ],
                dtype=bool
            )


            score = (
                tokens.astype(
                    np.float32
                )
                - 1.0
            ) / 2.0


            residual = (
                score
                -
                baselines[
                    "credit_item_mean"
                ][item]
            )


            residual[
                ~observed
            ] = 0.0


            value_list.append(
                residual
            )

            mask_list.append(
                observed
            )


        values = np.column_stack(
            value_list
        )

        masks = np.column_stack(
            mask_list
        )


        denominator = (
            masks.sum(
                axis=1
            )
        )


        credit_summary = np.divide(

            (
                values
                * masks
            ).sum(
                axis=1
            ),

            denominator,

            out=np.zeros(
                n,
                dtype=np.float32
            ),

            where=(
                denominator > 0
            )
        )


        credit_features.append(
            credit_summary
        )


        # Behaviour residuals

        for behaviour_index, token in enumerate(
            [1, 2, 3]
        ):

            value_list = []
            mask_list = []


            for item in selected_items:

                behaviour = np.asarray(
                    behaviour_tokens[
                        indices,
                        item
                    ]
                )


                observed = np.asarray(
                    interaction_mask[
                        indices,
                        item
                    ],
                    dtype=bool
                )


                residual = (

                    (
                        behaviour == token
                    ).astype(
                        np.float32
                    )

                    -

                    baselines[
                        "behaviour_item_rates"
                    ][
                        item,
                        behaviour_index
                    ]
                )


                residual[
                    ~observed
                ] = 0.0


                value_list.append(
                    residual
                )

                mask_list.append(
                    observed
                )


            values = np.column_stack(
                value_list
            )

            masks = np.column_stack(
                mask_list
            )


            denominator = (
                masks.sum(
                    axis=1
                )
            )


            summary = np.divide(

                (
                    values
                    * masks
                ).sum(
                    axis=1
                ),

                denominator,

                out=np.zeros(
                    n,
                    dtype=np.float32
                ),

                where=(
                    denominator > 0
                )
            )


            behaviour_features.append(
                summary
            )


        # Process residuals

        for channel in range(
            N_PROCESS_CHANNELS
        ):

            value_list = []
            mask_list = []


            for item in selected_items:

                x = np.asarray(
                    process_tensor[
                        indices,
                        item,
                        channel
                    ],
                    dtype=np.float32
                )


                observed = np.isfinite(
                    x
                )


                residual = np.zeros(
                    n,
                    dtype=np.float32
                )


                residual[
                    observed
                ] = (

                    x[
                        observed
                    ]

                    -

                    baselines[
                        "process_item_mean"
                    ][
                        item,
                        channel
                    ]

                ) / (

                    baselines[
                        "process_item_std"
                    ][
                        item,
                        channel
                    ]

                    + EPS
                )


                value_list.append(
                    residual
                )

                mask_list.append(
                    observed
                )


            values = np.column_stack(
                value_list
            )

            masks = np.column_stack(
                mask_list
            )


            denominator = (
                masks.sum(
                    axis=1
                )
            )


            summary = np.divide(

                (
                    values
                    * masks
                ).sum(
                    axis=1
                ),

                denominator,

                out=np.zeros(
                    n,
                    dtype=np.float32
                ),

                where=(
                    denominator > 0
                )
            )


            process_features.append(
                summary
            )


    return np.column_stack(
        [
            np.column_stack(
                credit_features
            ),

            np.column_stack(
                behaviour_features
            ),

            np.column_stack(
                process_features
            )
        ]
    ).astype(
        np.float32
    )


print(
    "Fold-specific item residualization utilities ready."
)

Fold-specific item residualization utilities ready.


In [ ]:
# Build all train-only resources for one outer fold

def build_fold_resources(
    outer_fold
):

    split = get_outer_split(
        outer_fold
    )


    train_idx = (
        split[
            "train_idx"
        ]
    )


    # 1. Context preprocessing

    student_pp = (
        ContextPreprocessor(

            context_registry_locked,

            common_student_vars
        )
        .fit(
            model_context,
            train_idx
        )
    )


    school_pp = (
        ContextPreprocessor(

            context_registry_locked,

            common_school_vars
        )
        .fit(
            model_context,
            train_idx
        )
    )


    # 2. Main process-branch robust scaling

    process_pp = (
        ProcessPreprocessor()
        .fit(
            process_tensor[
                train_idx
            ]
        )
    )


    # 3. Item-conditioned residualization parameters

    baselines = (
        learn_item_baselines(
            train_idx
        )
    )


    # 4. Training-system residual representation

    X_train_residual = (
        build_residual_summary_features(

            train_idx,

            baselines
        )
    )


    train_signatures = (
        design_signature_ids[
            train_idx
        ]
    )


    train_countries = (
        primary_meta.iloc[
            train_idx
        ]["CNT"]
        .to_numpy()
    )


    # 5. Country-equal design-signature profiles

    signature_profiles = np.zeros(
        (
            10,
            21
        ),
        dtype=np.float32
    )


    signature_country_counts = np.zeros(
        10,
        dtype=np.int64
    )


    for signature_id in range(
        10
    ):

        country_means = []


        for country in np.unique(
            train_countries
        ):

            mask = (

                (
                    train_signatures
                    == signature_id
                )

                &

                (
                    train_countries
                    == country
                )
            )


            if mask.any():

                country_means.append(

                    X_train_residual[
                        mask
                    ].mean(
                        axis=0
                    )
                )


        assert (
            len(
                country_means
            )
            > 0
        )


        country_means = np.vstack(
            country_means
        )


        signature_profiles[
            signature_id
        ] = (
            country_means.mean(
                axis=0
            )
        )


        signature_country_counts[
            signature_id
        ] = len(
            country_means
        )


    # All 10 signatures should occur in all 45
    # training education systems.
    assert (
        signature_country_counts.min()
        == 45
    )


    # 6. Convert 21-D profile -> 3 x 7 token offsets

    signature_credit = (
        signature_profiles[
            :,
            0:3
        ]
        .reshape(
            10,
            3,
            1
        )
    )


    signature_behaviour = (
        signature_profiles[
            :,
            3:12
        ]
        .reshape(
            10,
            3,
            3
        )
    )


    signature_process = (
        signature_profiles[
            :,
            12:21
        ]
        .reshape(
            10,
            3,
            3
        )
    )


    signature_token_profiles = (
        np.concatenate(
            [
                signature_credit,
                signature_behaviour,
                signature_process
            ],
            axis=2
        )
        .astype(
            np.float32
        )
    )


    assert (
        signature_token_profiles.shape
        == (10, 3, 7)
    )


    # 7. Training-country classifier IDs

    unique_train_countries = sorted(
        np.unique(
            train_countries
        ).tolist()
    )


    assert (
        len(
            unique_train_countries
        )
        == 45
    )


    country_to_train_id = {
        country: i

        for i, country
        in enumerate(
            unique_train_countries
        )
    }


    return {

        "outer_fold":
            outer_fold,

        "split":
            split,

        "student_pp":
            student_pp,

        "school_pp":
            school_pp,

        "process_pp":
            process_pp,

        "credit_item_mean":
            baselines[
                "credit_item_mean"
            ],

        "behaviour_item_rates":
            baselines[
                "behaviour_item_rates"
            ],

        "process_item_mean":
            baselines[
                "process_item_mean"
            ],

        "process_item_std":
            baselines[
                "process_item_std"
            ],

        "signature_profiles":
            signature_profiles,

        "signature_token_profiles":
            signature_token_profiles,

        "signature_country_counts":
            signature_country_counts,

        "country_to_train_id":
            country_to_train_id
    }


print(
    "build_fold_resources() ready."
)

build_fold_resources() ready.


In [ ]:
# Fold-0 reproducibility audit against locked artefacts

fold0_resources = build_fold_resources(
    0
)


# Previously locked Fold-0 artefacts

locked_credit_mean = np.load(
    CALIBRATION_DIR
    / "fold0_credit_item_mean.npy"
)

locked_behaviour_rates = np.load(
    CALIBRATION_DIR
    / "fold0_behaviour_item_rates.npy"
)

locked_process_mean = np.load(
    CALIBRATION_DIR
    / "fold0_process_item_mean.npy"
)

locked_process_std = np.load(
    CALIBRATION_DIR
    / "fold0_process_item_std.npy"
)

locked_signature_profiles = np.load(
    CALIBRATION_DIR
    / "fold0_signature_profiles.npy"
)


# Maximum absolute differences

comparison = {

    "credit_item_mean":
        np.max(
            np.abs(
                fold0_resources[
                    "credit_item_mean"
                ]
                -
                locked_credit_mean
            )
        ),

    "behaviour_item_rates":
        np.max(
            np.abs(
                fold0_resources[
                    "behaviour_item_rates"
                ]
                -
                locked_behaviour_rates
            )
        ),

    "process_item_mean":
        np.max(
            np.abs(
                fold0_resources[
                    "process_item_mean"
                ]
                -
                locked_process_mean
            )
        ),

    "process_item_std":
        np.max(
            np.abs(
                fold0_resources[
                    "process_item_std"
                ]
                -
                locked_process_std
            )
        ),

    "signature_profiles":
        np.max(
            np.abs(
                fold0_resources[
                    "signature_profiles"
                ]
                -
                locked_signature_profiles
            )
        )
}


print(
    "Fold-0 maximum absolute differences:"
)


for name, value in (
    comparison.items()
):

    print(
        f"{name:27s}",
        f"{value:.10f}"
    )


assert all(
    value < 1e-6

    for value
    in comparison.values()
)


print(
    "\nFold-0 generic-resource reproduction: PASS"
)


print(
    "Signature country counts:",
    fold0_resources[
        "signature_country_counts"
    ].tolist()
)


print(
    "Student cardinalities:",
    fold0_resources[
        "student_pp"
    ].cardinalities()
)

print(
    "School cardinalities:",
    fold0_resources[
        "school_pp"
    ].cardinalities()
)

Fold-0 maximum absolute differences:
credit_item_mean            0.0000000000
behaviour_item_rates        0.0000000000
process_item_mean           0.0000000000
process_item_std            0.0000000000
signature_profiles          0.0000000000

Fold-0 generic-resource reproduction: PASS
Signature country counts: [45, 45, 45, 45, 45, 45, 45, 45, 45, 45]
Student cardinalities: [4, 4, 5, 4]
School cardinalities: [8, 7, 9]


In [ ]:
# Build model-ready arrays for one outer fold only

def make_fold_split_data(
    indices,
    resources
):

    # Context

    student_ctx = (
        resources[
            "student_pp"
        ].transform(
            model_context.iloc[
                indices
            ]
        )
    )


    school_ctx = (
        resources[
            "school_pp"
        ].transform(
            model_context.iloc[
                indices
            ]
        )
    )


    # Process
    #
    # raw/log1p version:
    #   calibrated general branch
    #
    # fold-scaled version:
    #   main item-level process encoder

    process_raw = np.asarray(
        process_tensor[
            indices
        ],
        dtype=np.float32
    )


    process_values, process_mask = (
        resources[
            "process_pp"
        ].transform(
            process_raw
        )
    )


    # Metadata

    countries = (
        primary_meta.iloc[
            indices
        ]["CNT"]
        .to_numpy()
    )


    country_ids = np.array(
        [
            resources[
                "country_to_train_id"
            ].get(
                country,
                -1
            )

            for country
            in countries
        ],
        dtype=np.int64
    )


    booklet_ids = np.asarray(
        booklet_nuisance_ids[
            indices
        ],
        dtype=np.int64
    )


    signature_ids = (
        BOOKLET_TO_SIGNATURE[
            booklet_ids
        ]
        .astype(
            np.int64
        )
    )


    return {

        "credit_tokens":
            np.asarray(
                credit_tokens[
                    indices
                ],
                dtype=np.int64
            ),

        "behaviour_tokens":
            np.asarray(
                behaviour_tokens[
                    indices
                ],
                dtype=np.int64
            ),

        "performance_mask":
            np.asarray(
                performance_mask[
                    indices
                ],
                dtype=bool
            ),

        "interaction_mask":
            np.asarray(
                interaction_mask[
                    indices
                ],
                dtype=bool
            ),

        "design_mask":
            np.asarray(
                student_design_mask[
                    indices
                ],
                dtype=bool
            ),

        "process_raw":
            process_raw,

        "process_values":
            process_values,

        "process_mask":
            process_mask,

        "student_continuous":
            student_ctx[
                "continuous"
            ],

        "student_continuous_mask":
            student_ctx[
                "continuous_mask"
            ],

        "student_categorical":
            student_ctx[
                "categorical"
            ],

        "school_continuous":
            school_ctx[
                "continuous"
            ],

        "school_continuous_mask":
            school_ctx[
                "continuous_mask"
            ],

        "school_categorical":
            school_ctx[
                "categorical"
            ],

        "booklet_nuisance_id":
            booklet_ids,

        "design_signature_id":
            signature_ids,

        "country_train_id":
            country_ids,

        "CNT":
            countries,

        "CNTSCHID":
            primary_meta.iloc[
                indices
            ][
                "CNTSCHID"
            ].to_numpy(),

        "CNTSTUID":
            primary_meta.iloc[
                indices
            ][
                "CNTSTUID"
            ].to_numpy()
    }


def build_fold_bundle(
    outer_fold,
    resources=None
):

    if resources is None:

        resources = (
            build_fold_resources(
                outer_fold
            )
        )


    split = resources[
        "split"
    ]


    bundle = {

        "outer_fold":
            outer_fold,

        "resources":
            resources,

        "train":
            make_fold_split_data(
                split[
                    "train_idx"
                ],
                resources
            ),

        "validation":
            make_fold_split_data(
                split[
                    "validation_idx"
                ],
                resources
            ),

        "test":
            make_fold_split_data(
                split[
                    "test_idx"
                ],
                resources
            )
    }


    return bundle


print(
    "build_fold_bundle() ready."
)

build_fold_bundle() ready.


In [ ]:
# Generic CReST final Dataset

class CReSTFinalDataset(Dataset):

    def __init__(
        self,
        split_data
    ):

        self.d = split_data


    def __len__(
        self
    ):

        return len(
            self.d[
                "CNTSTUID"
            ]
        )


    def __getitem__(
        self,
        idx
    ):

        return {

            "credit_tokens":
                torch.as_tensor(
                    self.d[
                        "credit_tokens"
                    ][idx],
                    dtype=torch.long
                ),

            "behaviour_tokens":
                torch.as_tensor(
                    self.d[
                        "behaviour_tokens"
                    ][idx],
                    dtype=torch.long
                ),

            "performance_mask":
                torch.as_tensor(
                    self.d[
                        "performance_mask"
                    ][idx],
                    dtype=torch.bool
                ),

            "interaction_mask":
                torch.as_tensor(
                    self.d[
                        "interaction_mask"
                    ][idx],
                    dtype=torch.bool
                ),

            "design_mask":
                torch.as_tensor(
                    self.d[
                        "design_mask"
                    ][idx],
                    dtype=torch.bool
                ),

            "process_raw":
                torch.as_tensor(
                    self.d[
                        "process_raw"
                    ][idx],
                    dtype=torch.float32
                ),

            "process_values":
                torch.as_tensor(
                    self.d[
                        "process_values"
                    ][idx],
                    dtype=torch.float32
                ),

            "process_mask":
                torch.as_tensor(
                    self.d[
                        "process_mask"
                    ][idx],
                    dtype=torch.bool
                ),

            "student_continuous":
                torch.as_tensor(
                    self.d[
                        "student_continuous"
                    ][idx],
                    dtype=torch.float32
                ),

            "student_continuous_mask":
                torch.as_tensor(
                    self.d[
                        "student_continuous_mask"
                    ][idx],
                    dtype=torch.bool
                ),

            "student_categorical":
                torch.as_tensor(
                    self.d[
                        "student_categorical"
                    ][idx],
                    dtype=torch.long
                ),

            "school_continuous":
                torch.as_tensor(
                    self.d[
                        "school_continuous"
                    ][idx],
                    dtype=torch.float32
                ),

            "school_continuous_mask":
                torch.as_tensor(
                    self.d[
                        "school_continuous_mask"
                    ][idx],
                    dtype=torch.bool
                ),

            "school_categorical":
                torch.as_tensor(
                    self.d[
                        "school_categorical"
                    ][idx],
                    dtype=torch.long
                ),

            "booklet_nuisance_id":
                torch.as_tensor(
                    self.d[
                        "booklet_nuisance_id"
                    ][idx],
                    dtype=torch.long
                ),

            "design_signature_id":
                torch.as_tensor(
                    self.d[
                        "design_signature_id"
                    ][idx],
                    dtype=torch.long
                ),

            "country_train_id":
                torch.as_tensor(
                    self.d[
                        "country_train_id"
                    ][idx],
                    dtype=torch.long
                )
        }


print(
    "Generic CReSTFinalDataset ready."
)

Generic CReSTFinalDataset ready.


In [ ]:
# Generic Fold-0 bundle sanity check

fold0_bundle = build_fold_bundle(
    0,
    resources=fold0_resources
)


train_dataset_0 = CReSTFinalDataset(
    fold0_bundle[
        "train"
    ]
)

validation_dataset_0 = CReSTFinalDataset(
    fold0_bundle[
        "validation"
    ]
)

test_dataset_0 = CReSTFinalDataset(
    fold0_bundle[
        "test"
    ]
)


print(
    "Train:",
    len(
        train_dataset_0
    )
)

print(
    "Validation:",
    len(
        validation_dataset_0
    )
)

print(
    "Test:",
    len(
        test_dataset_0
    )
)


print(
    "\nProcess raw:",
    fold0_bundle[
        "train"
    ][
        "process_raw"
    ].shape
)

print(
    "Process scaled:",
    fold0_bundle[
        "train"
    ][
        "process_values"
    ].shape
)


print(
    "\nTraining country labels:",
    (
        fold0_bundle[
            "train"
        ][
            "country_train_id"
        ].min(),
        fold0_bundle[
            "train"
        ][
            "country_train_id"
        ].max()
    )
)


print(
    "Validation country labels unique:",
    np.unique(
        fold0_bundle[
            "validation"
        ][
            "country_train_id"
        ]
    )
)


print(
    "Test country labels unique:",
    np.unique(
        fold0_bundle[
            "test"
        ][
            "country_train_id"
        ]
    )
)


assert (
    len(
        train_dataset_0
    )
    == 99966
)

assert (
    len(
        validation_dataset_0
    )
    == 20949
)

assert (
    len(
        test_dataset_0
    )
    == 21649
)


assert (
    fold0_bundle[
        "train"
    ][
        "country_train_id"
    ].min()
    == 0
)

assert (
    fold0_bundle[
        "train"
    ][
        "country_train_id"
    ].max()
    == 44
)


assert (
    fold0_bundle[
        "validation"
    ][
        "country_train_id"
    ]
    == -1
).all()


assert (
    fold0_bundle[
        "test"
    ][
        "country_train_id"
    ]
    == -1
).all()


print(
    "\nFold-0 generic bundle: PASS"
)

print(
    "Validation/test country isolation: PASS"
)

print(
    "Test outcomes have NOT been evaluated."
)

Train: 99966
Validation: 20949
Test: 21649

Process raw: (99966, 32, 3)
Process scaled: (99966, 32, 3)

Training country labels: (0, 44)
Validation country labels unique: [-1]
Test country labels unique: [-1]

Fold-0 generic bundle: PASS
Validation/test country isolation: PASS
Test outcomes have NOT been evaluated.


In [ ]:
# Fold-specific calibrated general builder and utilities

class GradientReversalFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_grl * grad_output, None


def grad_reverse(x, lambda_grl=1.0):
    return GradientReversalFunction.apply(
        x,
        lambda_grl
    )


def masked_mean(
    x,
    mask,
    dim=1,
    eps=1e-8
):
    w = (
        mask
        .unsqueeze(-1)
        .to(x.dtype)
    )

    return (
        (x * w).sum(dim=dim)
        /
        w.sum(dim=dim).clamp_min(eps)
    )


class ContextEncoder(nn.Module):

    def __init__(
        self,
        n_continuous,
        categorical_cardinalities,
        d_model=128,
        cat_embedding_dim=16,
        dropout=0.1
    ):
        super().__init__()

        self.n_continuous = n_continuous

        self.categorical_embeddings = (
            nn.ModuleList(
                [
                    nn.Embedding(
                        cardinality,
                        cat_embedding_dim
                    )
                    for cardinality
                    in categorical_cardinalities
                ]
            )
        )

        input_dim = (
            2 * n_continuous
            +
            len(categorical_cardinalities)
            * cat_embedding_dim
        )

        self.encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                d_model
            ),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),

            nn.Linear(
                d_model,
                d_model
            ),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

    def forward(
        self,
        continuous,
        continuous_mask,
        categorical
    ):

        parts = []

        if self.n_continuous > 0:
            parts.extend(
                [
                    continuous,
                    continuous_mask.float()
                ]
            )

        for j, embedding in enumerate(
            self.categorical_embeddings
        ):
            parts.append(
                embedding(
                    categorical[:, j]
                )
            )

        x = torch.cat(
            parts,
            dim=1
        )

        return self.encoder(x)


class FoldCalibratedGeneralBuilder(nn.Module):
    """
    Fixed train-only nuisance calibration for one outer fold.

    Design signature is used only to select a fixed
    train-derived centering vector. It is never embedded
    or supplied as a predictive representation.
    """

    def __init__(
        self,
        item_ideation_ids,
        credit_item_mean,
        behaviour_item_rates,
        process_item_mean,
        process_item_std,
        signature_token_profiles
    ):
        super().__init__()

        self.register_buffer(
            "item_ideation_ids",
            torch.as_tensor(
                np.asarray(
                    item_ideation_ids
                ),
                dtype=torch.long
            )
        )

        self.register_buffer(
            "credit_item_mean",
            torch.as_tensor(
                credit_item_mean,
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "behaviour_item_rates",
            torch.as_tensor(
                behaviour_item_rates,
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "process_item_mean",
            torch.as_tensor(
                process_item_mean,
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "process_item_std",
            torch.as_tensor(
                process_item_std,
                dtype=torch.float32
            )
        )

        self.register_buffer(
            "signature_token_profiles",
            torch.as_tensor(
                signature_token_profiles,
                dtype=torch.float32
            )
        )

    @staticmethod
    def _masked_feature_mean(
        values,
        mask,
        dim=1
    ):
        mask_f = mask.to(
            values.dtype
        )

        numerator = (
            values
            * mask_f
        ).sum(
            dim=dim
        )

        denominator = (
            mask_f.sum(dim=dim)
            .clamp_min(1.0)
        )

        return (
            numerator
            /
            denominator
        )

    def forward(
        self,
        batch
    ):

        credit = batch[
            "credit_tokens"
        ]

        behaviour = batch[
            "behaviour_tokens"
        ]

        performance_mask = batch[
            "performance_mask"
        ]

        interaction_mask = batch[
            "interaction_mask"
        ]

        process_raw = batch[
            "process_raw"
        ]

        process_mask = batch[
            "process_mask"
        ]

        signature_id = batch[
            "design_signature_id"
        ]


        process_tokens = []
        availability = []


        for process_id in range(3):

            selector = (
                self.item_ideation_ids
                == process_id
            )

            # 1. Credit residual

            c = credit[
                :,
                selector
            ]

            c_valid = (
                performance_mask[
                    :,
                    selector
                ]
                &
                (c >= 1)
                &
                (c <= 3)
            )

            c_score = (
                c.float()
                - 1.0
            ) / 2.0

            c_residual = (
                c_score
                -
                self.credit_item_mean[
                    selector
                ].unsqueeze(0)
            )

            c_residual = torch.where(
                c_valid,
                c_residual,
                torch.zeros_like(
                    c_residual
                )
            )

            credit_summary = (
                self._masked_feature_mean(
                    c_residual,
                    c_valid,
                    dim=1
                )
            )


            # 2. Behaviour residuals

            b = behaviour[
                :,
                selector
            ]

            b_valid = (
                interaction_mask[
                    :,
                    selector
                ]
                &
                (b >= 1)
                &
                (b <= 3)
            )

            behaviour_summaries = []

            for behaviour_index, token in enumerate(
                [1, 2, 3]
            ):

                indicator = (
                    b == token
                ).float()

                residual = (
                    indicator
                    -
                    self.behaviour_item_rates[
                        selector,
                        behaviour_index
                    ].unsqueeze(0)
                )

                residual = torch.where(
                    b_valid,
                    residual,
                    torch.zeros_like(
                        residual
                    )
                )

                behaviour_summaries.append(
                    self._masked_feature_mean(
                        residual,
                        b_valid,
                        dim=1
                    )
                )


            # 3. Process residuals

            p_raw = process_raw[
                :,
                selector,
                :
            ]

            p_valid = process_mask[
                :,
                selector,
                :
            ]

            p_safe = torch.nan_to_num(
                p_raw,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )

            p_residual = (
                (
                    p_safe
                    -
                    self.process_item_mean[
                        selector,
                        :
                    ].unsqueeze(0)
                )
                /
                (
                    self.process_item_std[
                        selector,
                        :
                    ].unsqueeze(0)
                    +
                    1e-6
                )
            )

            p_residual = torch.where(
                p_valid,
                p_residual,
                torch.zeros_like(
                    p_residual
                )
            )

            process_summary = (
                self._masked_feature_mean(
                    p_residual,
                    p_valid,
                    dim=1
                )
            )


            # Assemble 7-D process token

            token_features = torch.stack(
                [
                    credit_summary,

                    behaviour_summaries[0],
                    behaviour_summaries[1],
                    behaviour_summaries[2],

                    process_summary[:, 0],
                    process_summary[:, 1],
                    process_summary[:, 2]
                ],
                dim=1
            )

            process_tokens.append(
                token_features
            )


            # Locked availability definition
            process_available = (
                interaction_mask[
                    :,
                    selector
                ].any(
                    dim=1
                )
            )

            availability.append(
                process_available
            )


        # [B, 3, 7]
        tokens = torch.stack(
            process_tokens,
            dim=1
        )

        availability = torch.stack(
            availability,
            dim=1
        )


        # Country-equal assessment-design centering

        nuisance_offset = (
            self.signature_token_profiles[
                signature_id
            ]
        )

        calibrated_tokens = (
            tokens
            -
            nuisance_offset
        )


        return (
            calibrated_tokens,
            availability
        )


print(
    "Fold-specific calibrated general builder ready."
)

Fold-specific calibrated general builder ready.


In [ ]:
# Generic locked CReST-Final architecture

class CReSTFinal(nn.Module):

    def __init__(
        self,
        resources,

        item_domain_ids,
        item_ideation_ids,

        n_train_countries=45,
        n_booklets=30,
        n_design_signatures=10,

        n_domains=4,
        n_processes=3,

        d_model=128,
        latent_dim=64,
        n_heads=8,
        n_layers=3,
        dropout=0.1
    ):
        super().__init__()

        self.n_items = 32
        self.n_domains = n_domains
        self.n_processes = n_processes

        self.d_model = d_model
        self.latent_dim = latent_dim


        # Fold-specific context configuration

        student_base_cardinalities = (
            resources[
                "student_pp"
            ].cardinalities()
        )

        school_base_cardinalities = (
            resources[
                "school_pp"
            ].cardinalities()
        )

        student_model_cardinalities = [
            c + 1
            for c in student_base_cardinalities
        ]

        school_model_cardinalities = [
            c + 1
            for c in school_base_cardinalities
        ]

        self.student_base_cardinalities = (
            student_base_cardinalities
        )

        self.school_base_cardinalities = (
            school_base_cardinalities
        )


        # Fixed item metadata

        self.register_buffer(
            "item_domain_ids",
            torch.as_tensor(
                np.asarray(
                    item_domain_ids
                ),
                dtype=torch.long
            )
        )

        self.register_buffer(
            "item_ideation_ids",
            torch.as_tensor(
                np.asarray(
                    item_ideation_ids
                ),
                dtype=torch.long
            )
        )


        # Fold-specific calibrated general builder

        self.general_builder = (
            FoldCalibratedGeneralBuilder(

                item_ideation_ids=(
                    item_ideation_ids
                ),

                credit_item_mean=(
                    resources[
                        "credit_item_mean"
                    ]
                ),

                behaviour_item_rates=(
                    resources[
                        "behaviour_item_rates"
                    ]
                ),

                process_item_mean=(
                    resources[
                        "process_item_mean"
                    ]
                ),

                process_item_std=(
                    resources[
                        "process_item_std"
                    ]
                ),

                signature_token_profiles=(
                    resources[
                        "signature_token_profiles"
                    ]
                )
            )
        )


        # Embeddings

        self.item_embedding = nn.Embedding(
            self.n_items,
            d_model
        )

        self.domain_embedding = nn.Embedding(
            n_domains,
            d_model
        )

        self.ideation_embedding = nn.Embedding(
            n_processes,
            d_model
        )

        self.credit_embedding = nn.Embedding(
            5,
            d_model
        )

        self.behaviour_embedding = nn.Embedding(
            4,
            d_model
        )


        # Item-level process branch

        self.process_projection = nn.Sequential(
            nn.Linear(
                6,
                d_model
            ),
            nn.GELU(),
            nn.LayerNorm(
                d_model
            )
        )


        # Performance Transformer

        performance_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,

                dropout=dropout,
                activation="gelu",

                batch_first=True,
                norm_first=True
            )
        )

        self.performance_encoder = (
            nn.TransformerEncoder(

                performance_layer,

                num_layers=n_layers,

                norm=nn.LayerNorm(
                    d_model
                )
            )
        )


        # Process Transformer

        process_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,

                dropout=dropout,
                activation="gelu",

                batch_first=True,
                norm_first=True
            )
        )

        self.process_encoder = (
            nn.TransformerEncoder(

                process_layer,

                num_layers=n_layers,

                norm=nn.LayerNorm(
                    d_model
                )
            )
        )


        # Cross-view fusion

        self.cross_attention = (
            nn.MultiheadAttention(

                embed_dim=d_model,
                num_heads=n_heads,

                dropout=dropout,

                batch_first=True
            )
        )

        self.cross_norm = nn.LayerNorm(
            d_model
        )


        # Design-robust general branch

        self.general_token_projection = (
            nn.Sequential(
                nn.Linear(
                    7,
                    d_model
                ),
                nn.GELU(),
                nn.LayerNorm(
                    d_model
                )
            )
        )

        general_layer = (
            nn.TransformerEncoderLayer(

                d_model=d_model,
                nhead=n_heads,

                dim_feedforward=2 * d_model,

                dropout=dropout,
                activation="gelu",

                batch_first=True,
                norm_first=True
            )
        )

        self.general_process_encoder = (
            nn.TransformerEncoder(

                general_layer,

                num_layers=1,

                norm=nn.LayerNorm(
                    d_model
                )
            )
        )

        self.general_pool_norm = (
            nn.LayerNorm(
                d_model
            )
        )

        self.creative_projection = (
            nn.Sequential(
                nn.Linear(
                    d_model,
                    latent_dim
                ),
                nn.LayerNorm(
                    latent_dim
                )
            )
        )


        # Context branch

        self.student_encoder = (
            ContextEncoder(

                n_continuous=len(
                    resources[
                        "student_pp"
                    ].continuous_vars
                ),

                categorical_cardinalities=(
                    student_model_cardinalities
                ),

                d_model=d_model,

                dropout=dropout
            )
        )

        self.school_encoder = (
            ContextEncoder(

                n_continuous=len(
                    resources[
                        "school_pp"
                    ].continuous_vars
                ),

                categorical_cardinalities=(
                    school_model_cardinalities
                ),

                d_model=d_model,

                dropout=dropout
            )
        )

        self.context_fusion = (
            nn.Sequential(
                nn.Linear(
                    2 * d_model,
                    d_model
                ),
                nn.GELU(),
                nn.LayerNorm(
                    d_model
                )
            )
        )

        self.context_projection = (
            nn.Sequential(
                nn.Linear(
                    d_model,
                    latent_dim
                ),
                nn.LayerNorm(
                    latent_dim
                )
            )
        )


        # Domain-specific branch

        self.domain_shared_projection = (
            nn.Sequential(
                nn.Linear(
                    d_model,
                    latent_dim
                ),
                nn.GELU(),
                nn.LayerNorm(
                    latent_dim
                )
            )
        )

        self.domain_specific_projections = (
            nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Linear(
                            d_model,
                            latent_dim
                        ),
                        nn.GELU(),
                        nn.LayerNorm(
                            latent_dim
                        )
                    )

                    for _ in range(
                        n_domains
                    )
                ]
            )
        )


        # Reconstruction heads

        self.credit_head = nn.Linear(
            d_model,
            3
        )

        self.process_head = nn.Linear(
            d_model,
            3
        )

        self.domain_latent_decoder = (
            nn.Sequential(
                nn.Linear(
                    latent_dim,
                    d_model
                ),
                nn.GELU(),
                nn.LayerNorm(
                    d_model
                )
            )
        )

        self.decomposition_credit_head = (
            nn.Linear(
                d_model,
                3
            )
        )


        # Context reconstruction

        total_continuous = (
            len(
                resources[
                    "student_pp"
                ].continuous_vars
            )
            +
            len(
                resources[
                    "school_pp"
                ].continuous_vars
            )
        )

        self.context_continuous_head = (
            nn.Linear(
                latent_dim,
                total_continuous
            )
            if total_continuous > 0
            else None
        )

        base_context_cardinalities = (
            student_base_cardinalities
            +
            school_base_cardinalities
        )

        self.context_categorical_heads = (
            nn.ModuleList(
                [
                    nn.Linear(
                        latent_dim,
                        cardinality
                    )

                    for cardinality
                    in base_context_cardinalities
                ]
            )
        )


        # Nuisance heads

        self.booklet_classifier = (
            nn.Sequential(
                nn.Linear(
                    latent_dim,
                    latent_dim
                ),
                nn.GELU(),
                nn.Linear(
                    latent_dim,
                    n_booklets
                )
            )
        )

        self.design_signature_classifier = (
            nn.Sequential(
                nn.Linear(
                    latent_dim,
                    latent_dim
                ),
                nn.GELU(),
                nn.Linear(
                    latent_dim,
                    n_design_signatures
                )
            )
        )

        self.context_country_classifier = (
            nn.Sequential(
                nn.Linear(
                    latent_dim,
                    latent_dim
                ),
                nn.GELU(),
                nn.Linear(
                    latent_dim,
                    n_train_countries
                )
            )
        )

        self.creative_country_classifier = (
            nn.Sequential(
                nn.Linear(
                    latent_dim,
                    latent_dim
                ),
                nn.GELU(),
                nn.Linear(
                    latent_dim,
                    n_train_countries
                )
            )
        )


    def encoder_metadata(
        self,
        batch_size
    ):

        meta = (
            self.domain_embedding(
                self.item_domain_ids
            )
            +
            self.ideation_embedding(
                self.item_ideation_ids
            )
        )

        return (
            meta
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )


    def decoder_metadata(
        self,
        batch_size,
        device_local
    ):

        item_ids = torch.arange(
            self.n_items,
            device=device_local
        )

        meta = (
            self.item_embedding(
                item_ids
            )
            +
            self.domain_embedding(
                self.item_domain_ids
            )
            +
            self.ideation_embedding(
                self.item_ideation_ids
            )
        )

        return (
            meta
            .unsqueeze(0)
            .expand(
                batch_size,
                -1,
                -1
            )
        )


    def forward(
        self,
        batch,
        lambda_booklet_grl=1.0,
        lambda_design_grl=1.0,
        lambda_country_grl=1.0
    ):

        credit = batch[
            "credit_tokens"
        ]

        behaviour = batch[
            "behaviour_tokens"
        ]

        interaction_mask = batch[
            "interaction_mask"
        ]

        process_values = batch[
            "process_values"
        ]

        process_mask = batch[
            "process_mask"
        ]


        batch_size = (
            credit.shape[0]
        )

        device_local = (
            credit.device
        )


        # A. Performance branch

        encoder_meta = (
            self.encoder_metadata(
                batch_size
            )
        )

        performance_tokens = (
            encoder_meta
            +
            self.credit_embedding(
                credit
            )
            +
            self.behaviour_embedding(
                behaviour
            )
        )

        performance_encoded = (
            self.performance_encoder(

                performance_tokens,

                src_key_padding_mask=(
                    ~interaction_mask
                )
            )
        )


        # B. Process branch

        process_input = torch.cat(
            [
                process_values,
                process_mask.float()
            ],
            dim=-1
        )

        process_tokens = (
            encoder_meta
            +
            self.process_projection(
                process_input
            )
        )

        process_item_mask = (
            process_mask.any(
                dim=-1
            )
        )

        safe_process_mask = (
            process_item_mask.clone()
        )

        no_process = (
            ~safe_process_mask.any(
                dim=1
            )
        )

        if no_process.any():

            safe_process_mask[
                no_process,
                0
            ] = True

            process_tokens = (
                process_tokens.clone()
            )

            process_tokens[
                no_process,
                0,
                :
            ] = 0.0


        process_encoded = (
            self.process_encoder(

                process_tokens,

                src_key_padding_mask=(
                    ~safe_process_mask
                )
            )
        )


        # C. Cross-view item representation

        cross_output, _ = (
            self.cross_attention(

                query=performance_encoded,

                key=process_encoded,

                value=process_encoded,

                key_padding_mask=(
                    ~safe_process_mask
                ),

                need_weights=False
            )
        )

        if no_process.any():

            cross_output = (
                cross_output.clone()
            )

            cross_output[
                no_process
            ] = 0.0


        creative_item_tokens = (
            self.cross_norm(
                performance_encoded
                +
                cross_output
            )
        )


        # D. Calibrated general branch

        (
            calibrated_general_tokens,
            general_available
        ) = self.general_builder(
            batch
        )


        general_tokens = (
            self.general_token_projection(
                calibrated_general_tokens
            )
        )


        process_ids = torch.arange(
            self.n_processes,
            device=device_local
        )

        general_tokens = (
            general_tokens
            +
            self.ideation_embedding(
                process_ids
            ).unsqueeze(0)
        )


        safe_general_available = (
            general_available.clone()
        )

        no_general = (
            ~safe_general_available.any(
                dim=1
            )
        )

        if no_general.any():

            safe_general_available[
                no_general,
                0
            ] = True

            general_tokens = (
                general_tokens.clone()
            )

            general_tokens[
                no_general,
                0,
                :
            ] = 0.0


        general_encoded = (
            self.general_process_encoder(

                general_tokens,

                src_key_padding_mask=(
                    ~safe_general_available
                )
            )
        )


        weights = (
            general_available
            .float()
            .unsqueeze(-1)
        )

        general_pooled = (
            (
                general_encoded
                * weights
            ).sum(
                dim=1
            )
            /
            weights.sum(
                dim=1
            ).clamp_min(1.0)
        )

        general_pooled = (
            self.general_pool_norm(
                general_pooled
            )
        )

        z_general = (
            self.creative_projection(
                general_pooled
            )
        )


        # E. Context branch

        student_context = (
            self.student_encoder(

                batch[
                    "student_continuous"
                ],

                batch[
                    "student_continuous_mask"
                ],

                batch[
                    "student_categorical"
                ]
            )
        )

        school_context = (
            self.school_encoder(

                batch[
                    "school_continuous"
                ],

                batch[
                    "school_continuous_mask"
                ],

                batch[
                    "school_categorical"
                ]
            )
        )

        context_fused = (
            self.context_fusion(
                torch.cat(
                    [
                        student_context,
                        school_context
                    ],
                    dim=1
                )
            )
        )

        z_context = (
            self.context_projection(
                context_fused
            )
        )


        # F. Domain-specific branch

        domain_pooled = []
        domain_available = []


        for domain_id in range(
            self.n_domains
        ):

            selector = (
                self.item_domain_ids
                .unsqueeze(0)
                == domain_id
            )

            valid = (
                interaction_mask
                &
                selector
            )

            available = (
                valid.any(
                    dim=1
                )
            )

            pooled = masked_mean(
                creative_item_tokens,
                valid,
                dim=1
            )

            pooled = torch.where(
                available.unsqueeze(-1),
                pooled,
                torch.zeros_like(
                    pooled
                )
            )

            domain_pooled.append(
                pooled
            )

            domain_available.append(
                available
            )


        domain_pooled = torch.stack(
            domain_pooled,
            dim=1
        )

        domain_available = torch.stack(
            domain_available,
            dim=1
        )


        z_domain_shared = (
            self.domain_shared_projection(
                domain_pooled
            )
        )

        z_domain_shared = torch.where(
            domain_available.unsqueeze(-1),
            z_domain_shared,
            torch.zeros_like(
                z_domain_shared
            )
        )


        specific_list = []

        for domain_id in range(
            self.n_domains
        ):

            z_specific = (
                self.domain_specific_projections[
                    domain_id
                ](
                    domain_pooled[
                        :,
                        domain_id,
                        :
                    ]
                )
            )

            z_specific = torch.where(
                domain_available[
                    :,
                    domain_id
                ].unsqueeze(-1),

                z_specific,

                torch.zeros_like(
                    z_specific
                )
            )

            specific_list.append(
                z_specific
            )


        z_domain_specific = torch.stack(
            specific_list,
            dim=1
        )

        z_domain_creative = (
            z_general.unsqueeze(1)
            +
            z_domain_specific
        )

        z_domain_creative = torch.where(
            domain_available.unsqueeze(-1),
            z_domain_creative,
            torch.zeros_like(
                z_domain_creative
            )
        )


        # G. Item reconstruction

        decoder_meta = (
            self.decoder_metadata(
                batch_size,
                device_local
            )
        )

        decoder_item_tokens = (
            creative_item_tokens
            +
            decoder_meta
        )

        credit_logits = (
            self.credit_head(
                decoder_item_tokens
            )
        )

        process_prediction = (
            self.process_head(
                decoder_item_tokens
            )
        )


        # H. Domain decomposition reconstruction

        domain_latent_per_item = (
            z_domain_creative.index_select(
                dim=1,
                index=self.item_domain_ids
            )
        )

        decomposition_tokens = (
            self.domain_latent_decoder(
                domain_latent_per_item
            )
            +
            decoder_meta
        )

        decomposition_credit_logits = (
            self.decomposition_credit_head(
                decomposition_tokens
            )
        )


        # I. Context reconstruction

        if (
            self.context_continuous_head
            is not None
        ):

            context_continuous_prediction = (
                self.context_continuous_head(
                    z_context
                )
            )

        else:

            context_continuous_prediction = (
                z_context.new_empty(
                    (
                        batch_size,
                        0
                    )
                )
            )


        context_categorical_logits = [
            head(
                z_context
            )
            for head
            in self.context_categorical_heads
        ]


        # J. Nuisance routing

        booklet_logits = (
            self.booklet_classifier(
                grad_reverse(
                    z_general,
                    lambda_booklet_grl
                )
            )
        )

        design_signature_logits = (
            self.design_signature_classifier(
                grad_reverse(
                    z_general,
                    lambda_design_grl
                )
            )
        )

        context_country_logits = (
            self.context_country_classifier(
                z_context
            )
        )

        creative_country_logits = (
            self.creative_country_classifier(
                grad_reverse(
                    z_general,
                    lambda_country_grl
                )
            )
        )


        return {

            "z_general":
                z_general,

            "z_creative":
                z_general,

            "z_context":
                z_context,

            "z_domain_shared":
                z_domain_shared,

            "z_domain_specific":
                z_domain_specific,

            "z_domain_creative":
                z_domain_creative,

            "domain_available":
                domain_available,

            "general_process_tokens":
                general_encoded,

            "general_process_available":
                general_available,

            "credit_logits":
                credit_logits,

            "process_prediction":
                process_prediction,

            "decomposition_credit_logits":
                decomposition_credit_logits,

            "context_continuous_prediction":
                context_continuous_prediction,

            "context_categorical_logits":
                context_categorical_logits,

            "booklet_logits":
                booklet_logits,

            "design_signature_logits":
                design_signature_logits,

            "context_country_logits":
                context_country_logits,

            "creative_country_logits":
                creative_country_logits
        }


print(
    "Generic locked CReSTFinal architecture ready."
)

Generic locked CReSTFinal architecture ready.


In [ ]:
# Fold-0 generic model reproducibility smoke test

set_global_seed(
    GLOBAL_SEED
)


model_check = CReSTFinal(

    resources=fold0_resources,

    item_domain_ids=item_domain_ids,

    item_ideation_ids=item_ideation_ids,

    n_train_countries=45,
    n_booklets=30,
    n_design_signatures=10,

    n_domains=4,
    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


smoke_loader = DataLoader(

    train_dataset_0,

    batch_size=64,

    shuffle=False,

    num_workers=0
)


smoke_batch = next(
    iter(
        smoke_loader
    )
)


smoke_batch = {
    key:
        value.to(device)

    for key, value
    in smoke_batch.items()
}


model_check.eval()


with torch.no_grad():

    smoke_output = model_check(

        smoke_batch,

        lambda_booklet_grl=0.0,

        lambda_design_grl=0.0,

        lambda_country_grl=0.0
    )


n_parameters = sum(
    p.numel()
    for p in model_check.parameters()
    if p.requires_grad
)


print(
    "Trainable parameters:",
    n_parameters
)

print(
    "z_general:",
    tuple(
        smoke_output[
            "z_general"
        ].shape
    )
)

print(
    "z_context:",
    tuple(
        smoke_output[
            "z_context"
        ].shape
    )
)

print(
    "credit_logits:",
    tuple(
        smoke_output[
            "credit_logits"
        ].shape
    )
)

print(
    "design logits:",
    tuple(
        smoke_output[
            "design_signature_logits"
        ].shape
    )
)


assert (
    n_parameters
    == 1577398
)

assert (
    smoke_output[
        "z_general"
    ].shape
    == (64, 64)
)

assert (
    smoke_output[
        "credit_logits"
    ].shape
    == (64, 32, 3)
)

assert (
    smoke_output[
        "design_signature_logits"
    ].shape
    == (64, 10)
)


print(
    "\nGeneric Fold-0 CReSTFinal smoke test: PASS"
)

Trainable parameters: 1577398
z_general: (64, 64)
z_context: (64, 64)
credit_logits: (64, 32, 3)
design logits: (64, 10)

Generic Fold-0 CReSTFinal smoke test: PASS


In [ ]:
# Lock final training protocol and SSL corruption

FINAL_TRAINING_PROTOCOL = {

    "max_epochs": 20,
    "early_stopping_patience": 4,
    "early_stopping_min_delta": 0.001,

    "batch_size": int(
        CONFIG["batch_size"]
    ),

    "learning_rate": 1e-4,
    "weight_decay": 1e-4,

    "gradient_clip_norm": 1.0,

    "lr_scheduler_factor": 0.5,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1e-6,

    "credit_mask_prob": 0.15,
    "process_mask_prob": 0.15,
    "context_mask_prob": 0.20,

    "sampler":
        "country-balanced 1/n_country",

    "checkpoint_metric":
        "country-equal validation reconstruction",

    "test_usage":
        "exactly once after checkpoint selection",

    "architecture":
        "CReST-Final LOCKED"
}


TRAINING_PROTOCOL_PATH = (
    RESULTS_DIR
    / "crest_final_training_protocol.json"
)


with open(
    TRAINING_PROTOCOL_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_TRAINING_PROTOCOL,
        f,
        indent=2
    )


LOSS_WEIGHTS = {

    "credit": 1.00,
    "process": 0.50,
    "decomposition_credit": 0.25,
    "context": 0.50,

    "decorrelation": 0.10,

    "booklet": 0.10,
    "design_signature": 0.10,

    "country_context": 0.10,
    "country_creative": 0.10,

    "global_variance": 0.05,

    "domain_alignment": 0.05,
    "domain_orthogonality": 0.05,
    "domain_variance": 0.02
}


def ensure_one_mask_per_eligible_row(
    selected_mask,
    eligible_mask
):

    needs_one = (
        eligible_mask.any(dim=1)
        &
        ~selected_mask.any(dim=1)
    )

    for row in torch.where(
        needs_one
    )[0]:

        positions = torch.where(
            eligible_mask[row]
        )[0]

        choice = positions[
            torch.randint(
                len(positions),
                (1,),
                device=positions.device
            )
        ]

        selected_mask[
            row,
            choice
        ] = True

    return selected_mask


def make_main_ssl_corruption(
    batch,
    credit_mask_prob=0.15,
    process_mask_prob=0.15
):

    corrupted = {
        key: value.clone()
        for key, value
        in batch.items()
    }


    # Credit

    credit_eligible = (
        batch[
            "performance_mask"
        ]
    )

    credit_ssl_mask = (
        credit_eligible
        &
        (
            torch.rand(
                credit_eligible.shape,
                device=credit_eligible.device
            )
            <
            credit_mask_prob
        )
    )

    credit_ssl_mask = (
        ensure_one_mask_per_eligible_row(
            credit_ssl_mask,
            credit_eligible
        )
    )

    credit_targets = (
        batch[
            "credit_tokens"
        ]
        - 1
    )

    corrupted[
        "credit_tokens"
    ][
        credit_ssl_mask
    ] = 4


    # Process

    process_item_eligible = (
        batch[
            "process_mask"
        ].any(
            dim=-1
        )
    )

    process_ssl_item_mask = (
        process_item_eligible
        &
        (
            torch.rand(
                process_item_eligible.shape,
                device=process_item_eligible.device
            )
            <
            process_mask_prob
        )
    )

    process_ssl_item_mask = (
        ensure_one_mask_per_eligible_row(
            process_ssl_item_mask,
            process_item_eligible
        )
    )

    process_targets = (
        batch[
            "process_values"
        ].clone()
    )

    expanded = (
        process_ssl_item_mask
        .unsqueeze(-1)
        .expand_as(
            batch[
                "process_values"
            ]
        )
    )

    process_reconstruction_mask = (
        expanded
        &
        batch[
            "process_mask"
        ]
    )

    corrupted[
        "process_values"
    ][expanded] = 0.0

    corrupted[
        "process_raw"
    ][expanded] = 0.0

    corrupted[
        "process_mask"
    ][expanded] = False


    return {

        "corrupted_batch":
            corrupted,

        "credit_targets":
            credit_targets,

        "credit_ssl_mask":
            credit_ssl_mask,

        "process_targets":
            process_targets,

        "process_reconstruction_mask":
            process_reconstruction_mask,

        "process_ssl_item_mask":
            process_ssl_item_mask
    }


def make_context_ssl_corruption(
    batch,
    resources,
    mask_prob=0.20
):

    corrupted = {
        key: value.clone()
        for key, value
        in batch.items()
    }

    continuous_targets = []
    continuous_masks = []

    categorical_targets = []
    categorical_masks = []


    # Continuous

    for prefix in [
        "student",
        "school"
    ]:

        values = batch[
            f"{prefix}_continuous"
        ]

        observed = batch[
            f"{prefix}_continuous_mask"
        ]

        selected = (
            observed
            &
            (
                torch.rand(
                    observed.shape,
                    device=observed.device
                )
                <
                mask_prob
            )
        )

        continuous_targets.append(
            values.clone()
        )

        continuous_masks.append(
            selected
        )

        corrupted[
            f"{prefix}_continuous"
        ][selected] = 0.0

        corrupted[
            f"{prefix}_continuous_mask"
        ][selected] = False


    # Categorical
    #
    # cardinalities() returns the number of base IDs.
    # That value itself is the extra SSL MASK token.

    categorical_specs = [

        (
            "student",
            resources[
                "student_pp"
            ].cardinalities()
        ),

        (
            "school",
            resources[
                "school_pp"
            ].cardinalities()
        )
    ]


    for prefix, mask_ids in (
        categorical_specs
    ):

        values = batch[
            f"{prefix}_categorical"
        ]

        observed = (
            values >= 2
        )

        selected = (
            observed
            &
            (
                torch.rand(
                    observed.shape,
                    device=observed.device
                )
                <
                mask_prob
            )
        )

        categorical_targets.append(
            values.clone()
        )

        categorical_masks.append(
            selected
        )

        for j, mask_id in enumerate(
            mask_ids
        ):

            feature_selected = (
                selected[:, j]
            )

            corrupted[
                f"{prefix}_categorical"
            ][
                feature_selected,
                j
            ] = int(
                mask_id
            )


    return {

        "corrupted_batch":
            corrupted,

        "continuous_targets":
            torch.cat(
                continuous_targets,
                dim=1
            ),

        "continuous_ssl_mask":
            torch.cat(
                continuous_masks,
                dim=1
            ),

        "categorical_targets":
            torch.cat(
                categorical_targets,
                dim=1
            ),

        "categorical_ssl_mask":
            torch.cat(
                categorical_masks,
                dim=1
            )
    }


print(
    "Final training protocol: LOCKED"
)

print(
    "Max epochs:",
    FINAL_TRAINING_PROTOCOL[
        "max_epochs"
    ]
)

print(
    "Early stopping patience:",
    FINAL_TRAINING_PROTOCOL[
        "early_stopping_patience"
    ]
)

print(
    "SSL corruption functions ready."
)

Final training protocol: LOCKED
Max epochs: 20
Early stopping patience: 4
SSL corruption functions ready.


In [ ]:
# Locked final CReST objective

def masked_credit_loss(
    logits,
    targets,
    mask
):

    if not mask.any():
        return logits.sum() * 0.0

    return F.cross_entropy(
        logits[mask],
        targets[mask]
    )


def masked_process_loss(
    prediction,
    targets,
    mask
):

    if not mask.any():
        return prediction.sum() * 0.0

    return F.smooth_l1_loss(
        prediction[mask],
        targets[mask],
        beta=1.0
    )


def context_continuous_loss(
    prediction,
    targets,
    mask
):

    if not mask.any():
        return prediction.sum() * 0.0

    return F.smooth_l1_loss(
        prediction[mask],
        targets[mask],
        beta=1.0
    )


def context_categorical_loss(
    logits_list,
    targets,
    mask
):

    losses = []

    for j, logits in enumerate(
        logits_list
    ):

        selected = (
            mask[:, j]
        )

        if selected.any():

            losses.append(
                F.cross_entropy(
                    logits[
                        selected
                    ],
                    targets[
                        selected,
                        j
                    ]
                )
            )

    if not losses:
        return logits_list[0].sum() * 0.0

    return torch.stack(
        losses
    ).mean()


def cross_latent_decorrelation(
    z_general,
    z_context,
    eps=1e-6
):

    zg = (
        z_general
        -
        z_general.mean(
            dim=0,
            keepdim=True
        )
    )

    zc = (
        z_context
        -
        z_context.mean(
            dim=0,
            keepdim=True
        )
    )

    zg = zg / (
        zg.std(
            dim=0,
            unbiased=False,
            keepdim=True
        )
        + eps
    )

    zc = zc / (
        zc.std(
            dim=0,
            unbiased=False,
            keepdim=True
        )
        + eps
    )

    corr = (
        zg.T
        @
        zc
    ) / zg.shape[0]

    return (
        corr
        .pow(2)
        .mean()
    )


def variance_loss(
    z,
    target_std=1.0
):

    std = torch.sqrt(
        z.var(
            dim=0,
            unbiased=False
        )
        + 1e-4
    )

    return F.relu(
        target_std
        -
        std
    ).mean()


def domain_alignment_loss(
    z_general,
    z_shared,
    available
):

    if not available.any():
        return z_shared.sum() * 0.0

    general = (
        z_general
        .unsqueeze(1)
        .expand_as(
            z_shared
        )
    )

    cosine = F.cosine_similarity(
        general[
            available
        ],
        z_shared[
            available
        ],
        dim=-1
    )

    return (
        1.0
        -
        cosine
    ).mean()


def domain_orthogonality_loss(
    z_general,
    z_specific,
    available
):

    if not available.any():
        return z_specific.sum() * 0.0

    general = (
        z_general
        .unsqueeze(1)
        .expand_as(
            z_specific
        )
    )

    cosine = F.cosine_similarity(
        general[
            available
        ],
        z_specific[
            available
        ],
        dim=-1
    )

    return (
        cosine
        .pow(2)
        .mean()
    )


def domain_variance_loss(
    z_specific,
    available,
    target_std=0.5
):

    losses = []

    for domain_id in range(
        z_specific.shape[1]
    ):

        valid = (
            available[
                :,
                domain_id
            ]
        )

        if valid.sum() < 2:
            continue

        std = torch.sqrt(
            z_specific[
                valid,
                domain_id
            ].var(
                dim=0,
                unbiased=False
            )
            + 1e-4
        )

        losses.append(
            F.relu(
                target_std
                -
                std
            ).mean()
        )

    if not losses:
        return z_specific.sum() * 0.0

    return torch.stack(
        losses
    ).mean()


def reconstruction_score_from_losses(
    losses
):

    return (
        losses["credit"]
        +
        LOSS_WEIGHTS["process"]
        *
        losses["process"]
        +
        LOSS_WEIGHTS[
            "decomposition_credit"
        ]
        *
        losses[
            "decomposition_credit"
        ]
        +
        LOSS_WEIGHTS["context"]
        *
        losses["context"]
    )


def compute_final_objective(
    model,
    batch,
    resources,

    lambda_booklet_grl=1.0,
    lambda_design_grl=1.0,
    lambda_country_grl=1.0
):

    main_ssl = (
        make_main_ssl_corruption(

            batch,

            credit_mask_prob=(
                FINAL_TRAINING_PROTOCOL[
                    "credit_mask_prob"
                ]
            ),

            process_mask_prob=(
                FINAL_TRAINING_PROTOCOL[
                    "process_mask_prob"
                ]
            )
        )
    )


    context_ssl = (
        make_context_ssl_corruption(

            main_ssl[
                "corrupted_batch"
            ],

            resources=resources,

            mask_prob=(
                FINAL_TRAINING_PROTOCOL[
                    "context_mask_prob"
                ]
            )
        )
    )


    output = model(

        context_ssl[
            "corrupted_batch"
        ],

        lambda_booklet_grl=(
            lambda_booklet_grl
        ),

        lambda_design_grl=(
            lambda_design_grl
        ),

        lambda_country_grl=(
            lambda_country_grl
        )
    )


    losses = {}


    losses["credit"] = (
        masked_credit_loss(

            output[
                "credit_logits"
            ],

            main_ssl[
                "credit_targets"
            ],

            main_ssl[
                "credit_ssl_mask"
            ]
        )
    )


    losses["process"] = (
        masked_process_loss(

            output[
                "process_prediction"
            ],

            main_ssl[
                "process_targets"
            ],

            main_ssl[
                "process_reconstruction_mask"
            ]
        )
    )


    losses[
        "decomposition_credit"
    ] = masked_credit_loss(

        output[
            "decomposition_credit_logits"
        ],

        main_ssl[
            "credit_targets"
        ],

        main_ssl[
            "credit_ssl_mask"
        ]
    )


    loss_context_cont = (
        context_continuous_loss(

            output[
                "context_continuous_prediction"
            ],

            context_ssl[
                "continuous_targets"
            ],

            context_ssl[
                "continuous_ssl_mask"
            ]
        )
    )


    loss_context_cat = (
        context_categorical_loss(

            output[
                "context_categorical_logits"
            ],

            context_ssl[
                "categorical_targets"
            ],

            context_ssl[
                "categorical_ssl_mask"
            ]
        )
    )


    losses["context"] = (
        0.5
        *
        (
            loss_context_cont
            +
            loss_context_cat
        )
    )


    losses["decorrelation"] = (
        cross_latent_decorrelation(

            output[
                "z_general"
            ],

            output[
                "z_context"
            ]
        )
    )


    losses["global_variance"] = (
        variance_loss(
            output[
                "z_general"
            ]
        )
    )


    losses["domain_alignment"] = (
        domain_alignment_loss(

            output[
                "z_general"
            ],

            output[
                "z_domain_shared"
            ],

            output[
                "domain_available"
            ]
        )
    )


    losses[
        "domain_orthogonality"
    ] = domain_orthogonality_loss(

        output[
            "z_general"
        ],

        output[
            "z_domain_specific"
        ],

        output[
            "domain_available"
        ]
    )


    losses["domain_variance"] = (
        domain_variance_loss(

            output[
                "z_domain_specific"
            ],

            output[
                "domain_available"
            ]
        )
    )


    losses["booklet"] = (
        F.cross_entropy(

            output[
                "booklet_logits"
            ],

            batch[
                "booklet_nuisance_id"
            ]
        )
    )


    losses["design_signature"] = (
        F.cross_entropy(

            output[
                "design_signature_logits"
            ],

            batch[
                "design_signature_id"
            ]
        )
    )


    country_targets = (
        batch[
            "country_train_id"
        ]
    )


    losses["country_context"] = (
        F.cross_entropy(

            output[
                "context_country_logits"
            ],

            country_targets
        )
    )


    losses["country_creative"] = (
        F.cross_entropy(

            output[
                "creative_country_logits"
            ],

            country_targets
        )
    )


    total_loss = sum(
        LOSS_WEIGHTS[name]
        *
        losses[name]

        for name
        in LOSS_WEIGHTS
    )


    return (
        total_loss,
        losses,
        output,
        main_ssl
    )


print(
    "Locked final objective ready."
)

Locked final objective ready.


In [ ]:
# Generic Fold-0 backward smoke test

BATCH_SIZE = (
    FINAL_TRAINING_PROTOCOL[
        "batch_size"
    ]
)


# Country-balanced Fold-0 sampler

train_countries_0 = (
    fold0_bundle[
        "train"
    ][
        "CNT"
    ]
)


unique_countries_0, counts_0 = (
    np.unique(
        train_countries_0,
        return_counts=True
    )
)


count_map_0 = {
    country: count

    for country, count
    in zip(
        unique_countries_0,
        counts_0
    )
}


sampling_weights_0 = np.array(
    [
        1.0
        /
        count_map_0[
            country
        ]

        for country
        in train_countries_0
    ],
    dtype=np.float64
)


smoke_sampler = (
    WeightedRandomSampler(

        weights=torch.as_tensor(
            sampling_weights_0,
            dtype=torch.double
        ),

        num_samples=len(
            train_dataset_0
        ),

        replacement=True,

        generator=(
            torch.Generator()
            .manual_seed(
                GLOBAL_SEED + 1700
            )
        )
    )
)


smoke_train_loader = (
    DataLoader(

        train_dataset_0,

        batch_size=BATCH_SIZE,

        sampler=smoke_sampler,

        shuffle=False,

        num_workers=0
    )
)


# Fresh generic model

set_global_seed(
    GLOBAL_SEED
)


smoke_model = CReSTFinal(

    resources=fold0_resources,

    item_domain_ids=item_domain_ids,

    item_ideation_ids=item_ideation_ids,

    n_train_countries=45,
    n_booklets=30,
    n_design_signatures=10,

    n_domains=4,
    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


smoke_batch = next(
    iter(
        smoke_train_loader
    )
)


smoke_batch = {
    key:
        value[:64]
        .to(device)

    for key, value
    in smoke_batch.items()
}


set_global_seed(
    GLOBAL_SEED + 1000
)


smoke_model.train()

smoke_model.zero_grad(
    set_to_none=True
)


total_loss, losses, output, ssl_info = (
    compute_final_objective(

        smoke_model,

        smoke_batch,

        resources=fold0_resources,

        lambda_booklet_grl=0.5,

        lambda_design_grl=0.5,

        lambda_country_grl=0.5
    )
)


assert torch.isfinite(
    total_loss
)


total_loss.backward()


def grad_norm(
    parameter
):

    if parameter.grad is None:
        return float("nan")

    return float(
        parameter.grad
        .detach()
        .norm()
        .cpu()
    )


critical_gradients = {

    "general_token_projection":
        grad_norm(
            smoke_model
            .general_token_projection[0]
            .weight
        ),

    "general_process_encoder":
        grad_norm(
            smoke_model
            .general_process_encoder
            .layers[0]
            .self_attn
            .in_proj_weight
        ),

    "creative_projection":
        grad_norm(
            smoke_model
            .creative_projection[0]
            .weight
        ),

    "item_embedding":
        grad_norm(
            smoke_model
            .item_embedding
            .weight
        ),

    "design_classifier":
        grad_norm(
            smoke_model
            .design_signature_classifier[0]
            .weight
        )
}


print(
    "Total loss:",
    round(
        float(
            total_loss.detach().cpu()
        ),
        6
    )
)


print(
    "\nCritical gradient norms:"
)


for name, value in (
    critical_gradients.items()
):

    print(
        f"{name:28s}",
        f"{value:.6f}"
    )


assert all(
    np.isfinite(value)
    and
    value > 0

    for value
    in critical_gradients.values()
)


print(
    "\nMasked credit cells:",
    int(
        ssl_info[
            "credit_ssl_mask"
        ].sum()
    )
)

print(
    "Masked process values:",
    int(
        ssl_info[
            "process_reconstruction_mask"
        ].sum()
    )
)


print(
    "\nGeneric final backward smoke test: PASS"
)

Total loss: 3.975422

Critical gradient norms:
general_token_projection     0.057460
general_process_encoder      0.076936
creative_projection          0.282325
item_embedding               0.065463
design_classifier            0.029512

Masked credit cells: 113
Masked process values: 379

Generic final backward smoke test: PASS


In [ ]:
# Deterministic country-equal validation

def grl_schedule(progress):

    return (
        2.0
        /
        (
            1.0
            +
            np.exp(
                -10.0 * progress
            )
        )
        -
        1.0
    )


@torch.no_grad()
def evaluate_country_equal_reconstruction(
    model,
    validation_dataset,
    validation_split_data,
    resources,
    seed
):

    model.eval()

    countries = (
        validation_split_data["CNT"]
    )

    unique_countries = sorted(
        np.unique(countries)
    )

    country_scores = {}


    # Preserve training RNG state.
    fork_devices = (
        [torch.cuda.current_device()]
        if torch.cuda.is_available()
        else []
    )


    with torch.random.fork_rng(
        devices=fork_devices
    ):

        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


        for country in unique_countries:

            local_idx = np.where(
                countries == country
            )[0]


            country_dataset = Subset(
                validation_dataset,
                local_idx.tolist()
            )


            loader = DataLoader(
                country_dataset,

                batch_size=(
                    FINAL_TRAINING_PROTOCOL[
                        "batch_size"
                    ]
                ),

                shuffle=False,
                num_workers=0
            )


            total_score = 0.0
            total_students = 0


            for batch in loader:

                batch = {
                    key:
                        value.to(device)

                    for key, value
                    in batch.items()
                }


                main_ssl = (
                    make_main_ssl_corruption(

                        batch,

                        credit_mask_prob=(
                            FINAL_TRAINING_PROTOCOL[
                                "credit_mask_prob"
                            ]
                        ),

                        process_mask_prob=(
                            FINAL_TRAINING_PROTOCOL[
                                "process_mask_prob"
                            ]
                        )
                    )
                )


                context_ssl = (
                    make_context_ssl_corruption(

                        main_ssl[
                            "corrupted_batch"
                        ],

                        resources=resources,

                        mask_prob=(
                            FINAL_TRAINING_PROTOCOL[
                                "context_mask_prob"
                            ]
                        )
                    )
                )


                output = model(

                    context_ssl[
                        "corrupted_batch"
                    ],

                    lambda_booklet_grl=0.0,
                    lambda_design_grl=0.0,
                    lambda_country_grl=0.0
                )


                val_losses = {}


                val_losses["credit"] = (
                    masked_credit_loss(

                        output[
                            "credit_logits"
                        ],

                        main_ssl[
                            "credit_targets"
                        ],

                        main_ssl[
                            "credit_ssl_mask"
                        ]
                    )
                )


                val_losses["process"] = (
                    masked_process_loss(

                        output[
                            "process_prediction"
                        ],

                        main_ssl[
                            "process_targets"
                        ],

                        main_ssl[
                            "process_reconstruction_mask"
                        ]
                    )
                )


                val_losses[
                    "decomposition_credit"
                ] = masked_credit_loss(

                    output[
                        "decomposition_credit_logits"
                    ],

                    main_ssl[
                        "credit_targets"
                    ],

                    main_ssl[
                        "credit_ssl_mask"
                    ]
                )


                loss_context_cont = (
                    context_continuous_loss(

                        output[
                            "context_continuous_prediction"
                        ],

                        context_ssl[
                            "continuous_targets"
                        ],

                        context_ssl[
                            "continuous_ssl_mask"
                        ]
                    )
                )


                loss_context_cat = (
                    context_categorical_loss(

                        output[
                            "context_categorical_logits"
                        ],

                        context_ssl[
                            "categorical_targets"
                        ],

                        context_ssl[
                            "categorical_ssl_mask"
                        ]
                    )
                )


                val_losses["context"] = (
                    0.5
                    *
                    (
                        loss_context_cont
                        +
                        loss_context_cat
                    )
                )


                score = (
                    reconstruction_score_from_losses(
                        val_losses
                    )
                )


                n_batch = (
                    batch[
                        "credit_tokens"
                    ].shape[0]
                )


                total_score += (
                    float(
                        score.cpu()
                    )
                    *
                    n_batch
                )

                total_students += (
                    n_batch
                )


            country_scores[
                country
            ] = (
                total_score
                /
                total_students
            )


    # Every education system receives equal weight.
    country_equal_score = float(
        np.mean(
            list(
                country_scores.values()
            )
        )
    )


    return (
        country_equal_score,
        country_scores
    )


print(
    "Deterministic country-equal validation ready."
)

Deterministic country-equal validation ready.


In [ ]:
# Final train_one_fold() function

import pickle


FINAL_RESOURCE_ROOT = (
    BASE_DIR
    / "processed"
    / "crest_final_7fold"
)

FINAL_RESOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


def save_fold_resources(
    outer_fold,
    resources
):

    fold_dir = (
        FINAL_RESOURCE_ROOT
        / f"fold_{outer_fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # Preprocessors are needed later to transform
    # the held-out test systems exactly as trained.
    with open(
        fold_dir / "preprocessors.pkl",
        "wb"
    ) as f:

        pickle.dump(
            {
                "student_pp":
                    resources[
                        "student_pp"
                    ],

                "school_pp":
                    resources[
                        "school_pp"
                    ],

                "process_pp":
                    resources[
                        "process_pp"
                    ],

                "country_to_train_id":
                    resources[
                        "country_to_train_id"
                    ]
            },
            f
        )


    np.save(
        fold_dir
        / "credit_item_mean.npy",

        resources[
            "credit_item_mean"
        ]
    )

    np.save(
        fold_dir
        / "behaviour_item_rates.npy",

        resources[
            "behaviour_item_rates"
        ]
    )

    np.save(
        fold_dir
        / "process_item_mean.npy",

        resources[
            "process_item_mean"
        ]
    )

    np.save(
        fold_dir
        / "process_item_std.npy",

        resources[
            "process_item_std"
        ]
    )

    np.save(
        fold_dir
        / "signature_profiles.npy",

        resources[
            "signature_profiles"
        ]
    )

    np.save(
        fold_dir
        / "signature_token_profiles.npy",

        resources[
            "signature_token_profiles"
        ]
    )


def train_one_fold(
    outer_fold,
    resources=None,
    train_split_data=None,
    validation_split_data=None
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"FINAL CReST TRAINING — OUTER FOLD {outer_fold}"
    )

    print(
        "=" * 72
    )


    # Resources

    if resources is None:

        resources = (
            build_fold_resources(
                outer_fold
            )
        )


    split = resources[
        "split"
    ]


    save_fold_resources(
        outer_fold,
        resources
    )


    # Build TRAIN + VALIDATION only
    #
    # No test transformation or evaluation here.

    if train_split_data is None:

        train_split_data = (
            make_fold_split_data(

                split[
                    "train_idx"
                ],

                resources
            )
        )


    if validation_split_data is None:

        validation_split_data = (
            make_fold_split_data(

                split[
                    "validation_idx"
                ],

                resources
            )
        )


    train_dataset = (
        CReSTFinalDataset(
            train_split_data
        )
    )


    validation_dataset = (
        CReSTFinalDataset(
            validation_split_data
        )
    )


    # Country-balanced training sampler

    train_countries = (
        train_split_data[
            "CNT"
        ]
    )


    unique_countries, counts = (
        np.unique(
            train_countries,
            return_counts=True
        )
    )


    assert (
        len(unique_countries)
        == 45
    )


    country_count_map = {
        country: count

        for country, count
        in zip(
            unique_countries,
            counts
        )
    }


    sampling_weights = np.array(
        [
            1.0
            /
            country_count_map[
                country
            ]

            for country
            in train_countries
        ],
        dtype=np.float64
    )


    sampler_seed = (
        GLOBAL_SEED
        +
        1700
        +
        outer_fold
    )


    train_sampler = (
        WeightedRandomSampler(

            weights=torch.as_tensor(
                sampling_weights,
                dtype=torch.double
            ),

            num_samples=len(
                train_dataset
            ),

            replacement=True,

            generator=(
                torch.Generator()
                .manual_seed(
                    sampler_seed
                )
            )
        )
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=(
            FINAL_TRAINING_PROTOCOL[
                "batch_size"
            ]
        ),

        sampler=train_sampler,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        )
    )


    # Fresh model

    model_seed = (
        GLOBAL_SEED
        +
        outer_fold
    )


    set_global_seed(
        model_seed
    )


    model = CReSTFinal(

        resources=resources,

        item_domain_ids=item_domain_ids,

        item_ideation_ids=item_ideation_ids,

        n_train_countries=45,
        n_booklets=30,
        n_design_signatures=10,

        n_domains=4,
        n_processes=3,

        d_model=int(
            CONFIG["d_model"]
        ),

        latent_dim=int(
            CONFIG["latent_dim"]
        ),

        n_heads=int(
            CONFIG["n_heads"]
        ),

        n_layers=int(
            CONFIG["n_layers"]
        ),

        dropout=float(
            CONFIG["dropout"]
        )
    ).to(device)


    # Optimizer / scheduler

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=(
            FINAL_TRAINING_PROTOCOL[
                "learning_rate"
            ]
        ),

        weight_decay=(
            FINAL_TRAINING_PROTOCOL[
                "weight_decay"
            ]
        )
    )


    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(

            optimizer,

            mode="min",

            factor=(
                FINAL_TRAINING_PROTOCOL[
                    "lr_scheduler_factor"
                ]
            ),

            patience=(
                FINAL_TRAINING_PROTOCOL[
                    "lr_scheduler_patience"
                ]
            ),

            min_lr=(
                FINAL_TRAINING_PROTOCOL[
                    "min_learning_rate"
                ]
            )
        )
    )


    # Paths

    fold_checkpoint_dir = (
        CHECKPOINT_ROOT
        / f"fold_{outer_fold}"
    )

    fold_checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    checkpoint_path = (
        fold_checkpoint_dir
        / "crest_final_best.pt"
    )


    history_path = (
        RESULTS_DIR
        / f"fold{outer_fold}_final_training_history.csv"
    )


    # Training state

    max_epochs = (
        FINAL_TRAINING_PROTOCOL[
            "max_epochs"
        ]
    )

    patience = (
        FINAL_TRAINING_PROTOCOL[
            "early_stopping_patience"
        ]
    )

    min_delta = (
        FINAL_TRAINING_PROTOCOL[
            "early_stopping_min_delta"
        ]
    )


    best_val = np.inf
    best_epoch = None

    epochs_without_improvement = 0

    history = []


    validation_seed = (
        GLOBAL_SEED
        +
        5000
        +
        outer_fold
    )


    # Epoch loop

    for epoch in range(
        1,
        max_epochs + 1
    ):

        epoch_start = time.time()

        model.train()


        running = {

            "total":
                0.0,

            "reconstruction":
                0.0,

            "booklet":
                0.0,

            "design_signature":
                0.0,

            "country_context":
                0.0,

            "country_creative":
                0.0
        }


        n_seen = 0


        for batch_idx, batch in enumerate(
            train_loader
        ):

            batch = {
                key:
                    value.to(device)

                for key, value
                in batch.items()
            }


            progress = (

                (
                    epoch - 1
                )

                +

                batch_idx
                /
                max(
                    len(
                        train_loader
                    ),
                    1
                )

            ) / max_epochs


            lambda_grl = float(
                grl_schedule(
                    progress
                )
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            (
                total_loss,
                losses,
                _,
                _
            ) = compute_final_objective(

                model,

                batch,

                resources=resources,

                lambda_booklet_grl=(
                    lambda_grl
                ),

                lambda_design_grl=(
                    lambda_grl
                ),

                lambda_country_grl=(
                    lambda_grl
                )
            )


            total_loss.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                max_norm=(
                    FINAL_TRAINING_PROTOCOL[
                        "gradient_clip_norm"
                    ]
                )
            )


            optimizer.step()


            n_batch = (
                batch[
                    "credit_tokens"
                ].shape[0]
            )


            reconstruction = (
                reconstruction_score_from_losses(
                    losses
                )
            )


            running["total"] += (
                float(
                    total_loss.detach().cpu()
                )
                *
                n_batch
            )

            running[
                "reconstruction"
            ] += (
                float(
                    reconstruction
                    .detach()
                    .cpu()
                )
                *
                n_batch
            )

            running["booklet"] += (
                float(
                    losses[
                        "booklet"
                    ]
                    .detach()
                    .cpu()
                )
                *
                n_batch
            )

            running[
                "design_signature"
            ] += (
                float(
                    losses[
                        "design_signature"
                    ]
                    .detach()
                    .cpu()
                )
                *
                n_batch
            )

            running[
                "country_context"
            ] += (
                float(
                    losses[
                        "country_context"
                    ]
                    .detach()
                    .cpu()
                )
                *
                n_batch
            )

            running[
                "country_creative"
            ] += (
                float(
                    losses[
                        "country_creative"
                    ]
                    .detach()
                    .cpu()
                )
                *
                n_batch
            )


            n_seen += (
                n_batch
            )


        # Validation

        (
            val_score,
            val_country_scores
        ) = evaluate_country_equal_reconstruction(

            model,

            validation_dataset,

            validation_split_data,

            resources,

            seed=validation_seed
        )


        scheduler.step(
            val_score
        )


        lr_now = (
            optimizer
            .param_groups[0]["lr"]
        )


        epoch_seconds = (
            time.time()
            -
            epoch_start
        )


        row = {

            "outer_fold":
                outer_fold,

            "epoch":
                epoch,

            "train_total":
                running["total"]
                / n_seen,

            "train_reconstruction":
                running[
                    "reconstruction"
                ]
                / n_seen,

            "train_booklet_ce":
                running["booklet"]
                / n_seen,

            "train_design_signature_ce":
                running[
                    "design_signature"
                ]
                / n_seen,

            "train_country_context_ce":
                running[
                    "country_context"
                ]
                / n_seen,

            "train_country_creative_ce":
                running[
                    "country_creative"
                ]
                / n_seen,

            "val_country_equal_reconstruction":
                val_score,

            "learning_rate":
                lr_now,

            "grl_end":
                lambda_grl,

            "epoch_seconds":
                epoch_seconds
        }


        history.append(
            row
        )


        # Persist after every epoch.
        pd.DataFrame(
            history
        ).to_csv(
            history_path,
            index=False
        )


        # Checkpoint / early stopping

        improved = (
            best_epoch is None
            or
            val_score
            <
            (
                best_val
                -
                min_delta
            )
        )


        if improved:

            best_val = (
                val_score
            )

            best_epoch = (
                epoch
            )

            epochs_without_improvement = 0


            torch.save(

                {
                    "outer_fold":
                        outer_fold,

                    "epoch":
                        epoch,

                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "val_country_equal_reconstruction":
                        val_score,

                    "val_country_scores":
                        val_country_scores,

                    "model_seed":
                        model_seed,

                    "sampler_seed":
                        sampler_seed,

                    "validation_seed":
                        validation_seed,

                    "training_protocol":
                        FINAL_TRAINING_PROTOCOL,

                    "architecture_status":
                        "LOCKED"
                },

                checkpoint_path
            )

        else:

            epochs_without_improvement += 1


        print(

            f"Fold {outer_fold} | "

            f"Epoch {epoch:02d} | "

            f"train={row['train_total']:.4f} | "

            f"recon={row['train_reconstruction']:.4f} | "

            f"val={val_score:.4f} | "

            f"bookletCE={row['train_booklet_ce']:.4f} | "

            f"designCE={row['train_design_signature_ce']:.4f} | "

            f"ctxCountryCE={row['train_country_context_ce']:.4f} | "

            f"creativeCountryCE={row['train_country_creative_ce']:.4f} | "

            f"GRL={lambda_grl:.3f} | "

            f"lr={lr_now:.1e} | "

            f"noImp={epochs_without_improvement}/{patience}"
        )


        if (
            epochs_without_improvement
            >= patience
        ):

            print(
                f"\nFold {outer_fold}: "
                f"early stopping at epoch {epoch}."
            )

            break


    # Final training summary

    summary = {

        "outer_fold":
            outer_fold,

        "best_epoch":
            best_epoch,

        "best_validation_reconstruction":
            best_val,

        "epochs_run":
            len(
                history
            ),

        "checkpoint":
            str(
                checkpoint_path
            ),

        "test_evaluated":
            False
    }


    summary_path = (
        RESULTS_DIR
        / f"fold{outer_fold}_training_summary.json"
    )


    with open(
        summary_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2
        )


    print(
        "\nBest epoch:",
        best_epoch
    )

    print(
        "Best validation reconstruction:",
        round(
            best_val,
            6
        )
    )

    print(
        "TEST evaluated: False"
    )

    print(
        "Checkpoint:",
        checkpoint_path
    )


    return summary

In [ ]:
# Run FINAL Fold-0 training only

fold0_final_summary = train_one_fold(

    outer_fold=0,

    resources=fold0_resources,

    train_split_data=(
        fold0_bundle[
            "train"
        ]
    ),

    validation_split_data=(
        fold0_bundle[
            "validation"
        ]
    )
)


FINAL CReST TRAINING — OUTER FOLD 0
Fold 0 | Epoch 01 | train=2.9528 | recon=1.6518 | val=1.4823 | bookletCE=3.0598 | designCE=2.3020 | ctxCountryCE=3.6836 | creativeCountryCE=3.7075 | GRL=0.244 | lr=1.0e-04 | noImp=0/4
Fold 0 | Epoch 02 | train=2.7240 | recon=1.4955 | val=1.4441 | bookletCE=2.9558 | designCE=2.3017 | ctxCountryCE=3.2438 | creativeCountryCE=3.6494 | GRL=0.462 | lr=1.0e-04 | noImp=0/4
Fold 0 | Epoch 03 | train=2.6739 | recon=1.4662 | val=1.4419 | bookletCE=2.8854 | designCE=2.3034 | ctxCountryCE=2.9688 | creativeCountryCE=3.8018 | GRL=0.635 | lr=1.0e-04 | noImp=0/4
Fold 0 | Epoch 04 | train=2.6540 | recon=1.4583 | val=1.4237 | bookletCE=2.8878 | designCE=2.3060 | ctxCountryCE=2.8478 | creativeCountryCE=3.7896 | GRL=0.761 | lr=1.0e-04 | noImp=0/4
Fold 0 | Epoch 05 | train=2.6442 | recon=1.4459 | val=1.4230 | bookletCE=2.8947 | designCE=2.3084 | ctxCountryCE=2.7783 | creativeCountryCE=3.8388 | GRL=0.848 | lr=1.0e-04 | noImp=1/4
Fold 0 | Epoch 06 | train=2.6296 | recon=1.

In [ ]:
# Final Fold-0 checkpoint frozen nuisance probe
# TRAIN + VALIDATION ONLY. Test remains untouched.

fold0_checkpoint_path = (
    CHECKPOINT_ROOT
    / "fold_0"
    / "crest_final_best.pt"
)


# Restore best final Fold-0 model

final_fold0_model = CReSTFinal(

    resources=fold0_resources,

    item_domain_ids=item_domain_ids,

    item_ideation_ids=item_ideation_ids,

    n_train_countries=45,
    n_booklets=30,
    n_design_signatures=10,

    n_domains=4,
    n_processes=3,

    d_model=int(
        CONFIG["d_model"]
    ),

    latent_dim=int(
        CONFIG["latent_dim"]
    ),

    n_heads=int(
        CONFIG["n_heads"]
    ),

    n_layers=int(
        CONFIG["n_layers"]
    ),

    dropout=float(
        CONFIG["dropout"]
    )
).to(device)


checkpoint = torch.load(
    fold0_checkpoint_path,
    map_location=device
)


final_fold0_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

final_fold0_model.eval()


print(
    "Restored epoch:",
    checkpoint["epoch"]
)

print(
    "Checkpoint validation:",
    checkpoint[
        "val_country_equal_reconstruction"
    ]
)


# Frozen latent extraction

@torch.no_grad()
def extract_probe_latents(
    model,
    dataset,
    batch_size=512
):

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )


    zg = []
    zc = []

    booklet = []
    signature = []


    for batch in loader:

        batch = {
            key:
                value.to(device)

            for key, value
            in batch.items()
        }


        output = model(

            batch,

            lambda_booklet_grl=0.0,
            lambda_design_grl=0.0,
            lambda_country_grl=0.0
        )


        zg.append(
            output[
                "z_general"
            ].cpu().numpy()
        )

        zc.append(
            output[
                "z_context"
            ].cpu().numpy()
        )

        booklet.append(
            batch[
                "booklet_nuisance_id"
            ].cpu().numpy()
        )

        signature.append(
            batch[
                "design_signature_id"
            ].cpu().numpy()
        )


    return {

        "z_general":
            np.concatenate(
                zg,
                axis=0
            ),

        "z_context":
            np.concatenate(
                zc,
                axis=0
            ),

        "booklet":
            np.concatenate(
                booklet,
                axis=0
            ),

        "design_signature":
            np.concatenate(
                signature,
                axis=0
            )
    }


probe_train = extract_probe_latents(
    final_fold0_model,
    train_dataset_0
)

probe_validation = extract_probe_latents(
    final_fold0_model,
    validation_dataset_0
)


def frozen_probe(
    X_train,
    y_train,
    X_validation,
    y_validation
):

    classifier = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            solver="lbfgs",
            class_weight="balanced",
            random_state=GLOBAL_SEED
        )
    )


    classifier.fit(
        X_train,
        y_train
    )


    pred = classifier.predict(
        X_validation
    )


    return {

        "accuracy":
            accuracy_score(
                y_validation,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation,
                pred
            ),

        "macro_f1":
            f1_score(
                y_validation,
                pred,
                average="macro"
            )
    }


probe_rows = []


for latent in [
    "z_general",
    "z_context"
]:

    for target, n_classes in [

        (
            "booklet",
            30
        ),

        (
            "design_signature",
            10
        )
    ]:

        result = frozen_probe(

            probe_train[
                latent
            ],

            probe_train[
                target
            ],

            probe_validation[
                latent
            ],

            probe_validation[
                target
            ]
        )


        probe_rows.append(
            {
                "latent":
                    latent,

                "target":
                    target,

                **result,

                "chance_BA":
                    1.0 / n_classes
            }
        )


fold0_final_probe_df = (
    pd.DataFrame(
        probe_rows
    )
)


print(
    fold0_final_probe_df.to_string(
        index=False
    )
)


fold0_final_probe_df.to_csv(

    RESULTS_DIR
    / "fold0_final_training_nuisance_probes.csv",

    index=False
)


print(
    "\nReference from 5-epoch architecture pilot:"
)

print(
    "z_general booklet BA  = 0.091034"
)

print(
    "z_general design BA   = 0.239282"
)

print(
    "\nFold-0 TEST remains untouched."
)

Restored epoch: 18
Checkpoint validation: 1.4023185071529571
   latent           target  accuracy  balanced_accuracy  macro_f1  chance_BA
z_general          booklet  0.054227           0.074423  0.040991   0.033333
z_general design_signature  0.220631           0.221296  0.211723   0.100000
z_context          booklet  0.007590           0.036456  0.008806   0.033333
z_context design_signature  0.097475           0.097608  0.094525   0.100000

Reference from 5-epoch architecture pilot:
z_general booklet BA  = 0.091034
z_general design BA   = 0.239282

Fold-0 TEST remains untouched.


In [ ]:
# Train final CReST models for outer folds 1-6
# TEST sets remain untouched.

remaining_fold_summaries = []


for outer_fold in range(
    1,
    7
):

    print(
        "\n\n"
        + "#" * 78
    )

    print(
        f"STARTING OUTER FOLD {outer_fold}"
    )

    print(
        "#" * 78
    )


    # Build fold-specific train-only resources

    resources = (
        build_fold_resources(
            outer_fold
        )
    )


    split = resources[
        "split"
    ]


    # Build TRAIN + VALIDATION only
    #
    # IMPORTANT:
    # test split is NOT transformed here.

    train_split_data = (
        make_fold_split_data(

            split[
                "train_idx"
            ],

            resources
        )
    )


    validation_split_data = (
        make_fold_split_data(

            split[
                "validation_idx"
            ],

            resources
        )
    )


    # Train

    summary = train_one_fold(

        outer_fold=outer_fold,

        resources=resources,

        train_split_data=(
            train_split_data
        ),

        validation_split_data=(
            validation_split_data
        )
    )


    remaining_fold_summaries.append(
        summary
    )


    # Free fold-specific RAM before next fold

    del train_split_data
    del validation_split_data
    del resources

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    print(
        f"\nOUTER FOLD {outer_fold} COMPLETE"
    )

    print(
        "TEST evaluated: False"
    )


# Save combined training summary

all_training_summaries = [
    fold0_final_summary,
    *remaining_fold_summaries
]


all_training_summary_df = (
    pd.DataFrame(
        all_training_summaries
    )
)


all_training_summary_df.to_csv(

    RESULTS_DIR
    / "crest_final_7fold_training_summary.csv",

    index=False
)


print(
    "\n"
    + "=" * 78
)

print(
    "ALL 7 FINAL TRAINING FOLDS COMPLETE"
)

print(
    "=" * 78
)


print(
    all_training_summary_df[
        [
            "outer_fold",
            "best_epoch",
            "best_validation_reconstruction",
            "epochs_run",
            "test_evaluated"
        ]
    ].to_string(
        index=False
    )
)


assert (
    all_training_summary_df[
        "test_evaluated"
    ]
    .eq(False)
    .all()
)


print(
    "\nAll held-out TEST systems remain untouched: PASS"
)



##############################################################################
STARTING OUTER FOLD 1
##############################################################################

FINAL CReST TRAINING — OUTER FOLD 1
Fold 1 | Epoch 01 | train=2.9909 | recon=1.6868 | val=1.5496 | bookletCE=3.1253 | designCE=2.3051 | ctxCountryCE=3.6908 | creativeCountryCE=3.6771 | GRL=0.244 | lr=1.0e-04 | noImp=0/4
Fold 1 | Epoch 02 | train=2.7212 | recon=1.4964 | val=1.4974 | bookletCE=2.9189 | designCE=2.3007 | ctxCountryCE=3.2389 | creativeCountryCE=3.6612 | GRL=0.462 | lr=1.0e-04 | noImp=0/4
Fold 1 | Epoch 03 | train=2.6782 | recon=1.4693 | val=1.4816 | bookletCE=2.8649 | designCE=2.3033 | ctxCountryCE=2.9869 | creativeCountryCE=3.8105 | GRL=0.635 | lr=1.0e-04 | noImp=0/4
Fold 1 | Epoch 04 | train=2.6552 | recon=1.4587 | val=1.4750 | bookletCE=2.8501 | designCE=2.3056 | ctxCountryCE=2.8578 | creativeCountryCE=3.8372 | GRL=0.761 | lr=1.0e-04 | noImp=0/4
Fold 1 | Epoch 05 | train=2.6325 | recon=1.44


FINAL CReST TRAINING — OUTER FOLD 3
Fold 3 | Epoch 01 | train=3.0002 | recon=1.6864 | val=1.5036 | bookletCE=3.1116 | designCE=2.3045 | ctxCountryCE=3.7457 | creativeCountryCE=3.7167 | GRL=0.244 | lr=1.0e-04 | noImp=0/4
Fold 3 | Epoch 02 | train=2.7282 | recon=1.4943 | val=1.4511 | bookletCE=2.8960 | designCE=2.2987 | ctxCountryCE=3.3673 | creativeCountryCE=3.6548 | GRL=0.462 | lr=1.0e-04 | noImp=0/4
Fold 3 | Epoch 03 | train=2.6795 | recon=1.4674 | val=1.4367 | bookletCE=2.8505 | designCE=2.3037 | ctxCountryCE=3.0472 | creativeCountryCE=3.8025 | GRL=0.635 | lr=1.0e-04 | noImp=0/4
Fold 3 | Epoch 04 | train=2.6790 | recon=1.4588 | val=1.4291 | bookletCE=2.9877 | designCE=2.3095 | ctxCountryCE=2.9268 | creativeCountryCE=3.7909 | GRL=0.761 | lr=1.0e-04 | noImp=0/4
Fold 3 | Epoch 05 | train=2.6458 | recon=1.4452 | val=1.4344 | bookletCE=2.8899 | designCE=2.3061 | ctxCountryCE=2.8593 | creativeCountryCE=3.7920 | GRL=0.848 | lr=1.0e-04 | noImp=1/4
Fold 3 | Epoch 06 | train=2.6438 | recon=1.

Fold 5 | Epoch 03 | train=2.6793 | recon=1.4715 | val=1.4772 | bookletCE=2.8609 | designCE=2.3023 | ctxCountryCE=2.9817 | creativeCountryCE=3.8143 | GRL=0.635 | lr=1.0e-04 | noImp=0/4
Fold 5 | Epoch 04 | train=2.6709 | recon=1.4598 | val=1.4764 | bookletCE=2.9598 | designCE=2.3100 | ctxCountryCE=2.8610 | creativeCountryCE=3.8047 | GRL=0.761 | lr=1.0e-04 | noImp=1/4
Fold 5 | Epoch 05 | train=2.6495 | recon=1.4536 | val=1.4697 | bookletCE=2.8765 | designCE=2.3079 | ctxCountryCE=2.8039 | creativeCountryCE=3.8213 | GRL=0.848 | lr=1.0e-04 | noImp=0/4
Fold 5 | Epoch 06 | train=2.6409 | recon=1.4455 | val=1.4599 | bookletCE=2.9077 | designCE=2.3066 | ctxCountryCE=2.7643 | creativeCountryCE=3.8294 | GRL=0.905 | lr=1.0e-04 | noImp=0/4
Fold 5 | Epoch 07 | train=2.6229 | recon=1.4467 | val=1.4702 | bookletCE=2.8204 | designCE=2.3041 | ctxCountryCE=2.7413 | creativeCountryCE=3.8042 | GRL=0.941 | lr=1.0e-04 | noImp=1/4
Fold 5 | Epoch 08 | train=2.6140 | recon=1.4414 | val=1.4629 | bookletCE=2.8116 